# ReServe: greedy LIS order and intermediate FP16/AWQ profiles

**Run this next, in a fresh Kaggle T4 notebook.** The switching-cost run is complete; do not rerun it here.

This notebook first computes which layers to quantize, freezes the complete order, then measures latency
and memory at representative points on that path. All work runs in the same Kaggle notebook.

| Part | Default work |
|---|---|
| Local scores | LTS/LRS for all 28 layers on original FP16 inputs |
| Greedy order | 406 conditional candidate evaluations; MDS recomputed after every selected layer |
| State validation | Native retained-KV drop/restore along the frozen order |
| Memory | All 29 configurations, keeping tensor backing and allocator blocks separate |
| Operation profiles | Fitting q=1,7,14,21; held-out q=10,18; targeted q=0/28 anchors |
| Validation | Held-out operation timing interpolation and final exact FP16 restoration |

**Where:** Kaggle → create a new notebook → File → Import Notebook → upload this `.ipynb`.
Enable **Internet** and **GPU T4**. Only GPU 0 is used even if two T4s are exposed. Use a fresh kernel with
the same Kaggle Python 3.13 environment as the previous successful run.

**How:** Run Cell 1. If it installs packages, restart the kernel and run from the top. Then run every later
cell in order. The compatibility bundle and prior run provenance are embedded; WikiText downloads automatically.
No extra input upload is needed. Leave the default configuration for the complete initial run.

This is a heavier run than switching profiling. Cases and greedy rounds are checkpointed; keep the writable
output folder if you need to resume after restarting the kernel. Download the final `lis_intermediate_*.zip`
and provide the full ZIP for review before using its order/cost tables in the simulator.


## 1. Install/check the matched environment

Run this first in every new kernel, even if packages are already installed: it defines the version checker
used later. The serving stack is unchanged. PyArrow is only a CPU reader for pinned WikiText Parquet files.
If installation occurs, the cell intentionally stops and asks for a kernel restart.


In [1]:
import importlib.metadata as md
import subprocess
import sys

INSTALL_IF_NEEDED = True
PINS = {
    "torch": "2.6.0", "triton": "3.2.0", "transformers": "4.51.3",
    "autoawq": "0.2.9", "accelerate": "1.6.0", "safetensors": "0.5.3",
    "huggingface_hub": "0.30.2", "tokenizers": "0.21.1", "numpy": "2.1.3", "pyarrow": "20.0.0",
}

def installed_version(name):
    try:
        return md.version(name)
    except md.PackageNotFoundError:
        return None

mismatch = {k: installed_version(k) for k, v in PINS.items()
            if (installed_version(k) or "").split("+")[0] != v}
cuda_build_wrong = "+cu124" not in (installed_version("torch") or "")
print({k: installed_version(k) for k in PINS})
if mismatch or cuda_build_wrong:
    if not INSTALL_IF_NEEDED:
        raise RuntimeError("Recorded packages differ. Enable installation and restart after it finishes.")
    if "torch" in mismatch or cuda_build_wrong:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
            "torch==2.6.0", "torchvision==0.21.0", "torchaudio==2.6.0",
            "--index-url", "https://download.pytorch.org/whl/cu124"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
        *[f"{k}=={v}" for k, v in PINS.items() if k != "torch"]])
    raise RuntimeError("Installation complete. RESTART the Kaggle kernel, then run from the top.")
print("Recorded versions match. Continue to Cell 2.")


{'torch': '2.11.0+cpu', 'triton': None, 'transformers': '5.20.2', 'autoawq': None, 'accelerate': '1.15.0', 'safetensors': '0.8.0', 'huggingface_hub': '1.33.0', 'tokenizers': '0.23.3', 'numpy': '2.1.3', 'pyarrow': '23.0.1'}


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
pathos 0.3.5 requires dill>=0.4.1, but you have dill 0.3.8 which is incompatible.
pathos 0.3.5 requires multiprocess>=0.70.19, but you have multiprocess 0.70.16 which is incompatible.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.
litellm 1.109.1 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.30.2 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.3.0 which is incompatible.
bigframes 2.51.0 requires pyarrow>=23.0.1, but you have pyarrow 20.0.0 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.30.2 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,

RuntimeError: Installation complete. RESTART the Kaggle kernel, then run from the top.

## 2. Provenance and settings

Run the collapsed provenance cell unchanged. Then run the settings/helpers cell.
For a new run leave `RESUME_DIRECTORY` empty. To resume after a kernel restart, set it to the existing
writable run folder and rerun from the top without changing the configuration or helper code.

Default calibration: **8 × 2,048-token WikiText examples, 32 fixed token positions per example**.
Defaults profile the matched batch/context grid with 3 warmups, 10 timing trials and 3 memory trials per case.


In [ ]:
CODE_SHA256='9f02a3a7c3fc955117de8d56a80852c55107cdd261e74a168dc81ed79bfd576b'
COMPATIBILITY_ZIP_BASE64='...'
PRIOR_SWITCHING_METADATA_JSON='{\n  "schema_version": 2,\n  "run_id": "20261127T211843_842337Z",\n  "configuration_sha256": "d7ce55933ae2ea7817232fa40fb0ec3b2ff9d6d95a023c09a17466fe7f54e4d8",\n  "settings": {\n    "schema_version": 2,\n    "profiler_revision": "2026-11-27.cleanup-ledger.2",\n    "seed": 20261127,\n    "warmups": 3,\n    "trials": 10,\n    "memory_probes": 1,\n    "idle_layers": [\n      0,\n      1,\n      2,\n      3,\n      4,\n      5,\n      6,\n      7,\n      8,\n      9,\n      10,\n      11,\n      12,\n      13,\n      14,\n      15,\n      16,\n      17,\n      18,\n      19,\n      20,\n      21,\n      22,\n      23,\n      24,\n      25,\n      26,\n      27\n    ],\n    "activity_layers": [\n      0,\n      13,\n      27\n    ],\n    "activity_warmups": 3,\n    "activity_trials": 5,\n    "activity_shapes": [\n      {\n        "phase": "prefill",\n        "batch": 1,\n        "length": 256\n      },\n      {\n        "phase": "prefill",\n        "batch": 1,\n        "length": 1024\n      },\n      {\n        "phase": "decode",\n        "batch": 1,\n        "length": 512\n      }\n    ],\n    "host_policy": "all_variants_pinned_no_silent_fallback",\n    "allocation_layout": "whole_layer_allocate_copy_commit_release",\n    "kernel": "native_autoawq_gemm_triton_split_k_8_sdpa",\n    "final_LIS_order": false\n  },\n  "models": {\n    "fp16": {\n      "repo": "Qwen/Qwen2.5-1.5B-Instruct",\n      "revision": "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"\n    },\n    "awq": {\n      "repo": "Qwen/Qwen2.5-1.5B-Instruct-AWQ",\n      "revision": "c5b72040769bfde6c5b562ae890f9657b0803f79"\n    }\n  },\n  "code_sha256": "3f3dec822e48b55346dcc1ce2a04d66db4c2ba91de24b16a0d07bf2c12ed57f9",\n  "sessions": [\n    {\n      "session_id": "20261127T211843_842337Z",\n      "start_utc": "2026-11-27T21:18:43.843571+00:00",\n      "environment": {\n        "python": "3.13.16",\n        "packages": {\n          "torch": "2.6.0+cu124",\n          "triton": "3.2.0",\n          "transformers": "4.51.3",\n          "autoawq": "0.2.9",\n          "accelerate": "1.6.0",\n          "safetensors": "0.5.3",\n          "huggingface_hub": "0.30.2",\n          "tokenizers": "0.21.1",\n          "numpy": "2.1.3"\n        },\n        "torch": "2.6.0+cu124",\n        "cuda_runtime": "12.4",\n        "gpu": "Tesla T4",\n        "gpu_total_bytes": 15636037632,\n        "visible_gpus": 2,\n        "selected_gpu": 0,\n        "allocator_backend": "native",\n        "allocator_env": {\n          "PYTORCH_CUDA_ALLOC_CONF": null,\n          "CUDA_VISIBLE_DEVICES": null\n        },\n        "attention": "sdpa",\n        "fused_modules": false,\n        "split_k": 8,\n        "optional_extension_present_before_override": false,\n        "gemm_forward_source_sha256": "339a1590e6e2e5bc672e2ecb7cf4fb1c04dd6a789d018d68d328205f9963789b",\n        "copy_stream_id": 1312512176,\n        "compute_stream_id": 0\n      }\n    }\n  ],\n  "checks": {\n    "environment": {\n      "status": "PASS",\n      "setup_or_stage_wall_seconds": 18.207434200999955,\n      "details": {\n        "python": "3.13.16",\n        "packages": {\n          "torch": "2.6.0+cu124",\n          "triton": "3.2.0",\n          "transformers": "4.51.3",\n          "autoawq": "0.2.9",\n          "accelerate": "1.6.0",\n          "safetensors": "0.5.3",\n          "huggingface_hub": "0.30.2",\n          "tokenizers": "0.21.1",\n          "numpy": "2.1.3"\n        },\n        "torch": "2.6.0+cu124",\n        "cuda_runtime": "12.4",\n        "gpu": "Tesla T4",\n        "gpu_total_bytes": 15636037632,\n        "visible_gpus": 2,\n        "selected_gpu": 0,\n        "allocator_backend": "native",\n        "allocator_env": {\n          "PYTORCH_CUDA_ALLOC_CONF": null,\n          "CUDA_VISIBLE_DEVICES": null\n        },\n        "attention": "sdpa",\n        "fused_modules": false,\n        "split_k": 8,\n        "optional_extension_present_before_override": false,\n        "gemm_forward_source_sha256": "339a1590e6e2e5bc672e2ecb7cf4fb1c04dd6a789d018d68d328205f9963789b",\n        "copy_stream_id": 1312512176,\n        "compute_stream_id": 0\n      }\n    },\n    "evidence": {\n      "status": "PASS",\n      "setup_or_stage_wall_seconds": 25.58809285399991,\n      "details": {\n        "compatibility_sha256": "46f0f521cfec25c165f56a5f5faeae31550940b9a9a107440f3da3f7bf181649",\n        "variant_manifest_sha256": "8cbdec3df8ecc9a86c0ea9b78206281401408420f350220d9f91d41aefa3549d",\n        "boundary": "complete decoder module including both companion norms",\n        "tokenizer_and_chat_template": "pinned FP16 tokenizer for every configuration"\n      }\n    },\n    "host_setup": {\n      "status": "PASS",\n      "setup_or_stage_wall_seconds": 18.01662604799992,\n      "details": {\n        "pinned_tensor_bytes": 3301652480,\n        "complete_variant_sets": 2,\n        "gpu_models": 1\n      }\n    },\n    "executor_check": {\n      "status": "PASS",\n      "setup_or_stage_wall_seconds": 25.022022634999985,\n      "details": {\n        "native_retained_KV_cycle": "all 28 down, token appended, all 28 up, token appended",\n        "old_cached_values_preserved": true,\n        "original_tensors_exact": true,\n        "serial_awq_repeat_max_absolute_difference": 0.0,\n        "serial_comparison_atol": 0.02,\n        "serial_comparison_rtol": 0.002,\n        "fresh_cache_fp16_max_absolute_difference": 0.0,\n        "scope": "Executor continuity and stored-state identity; no semantic-quality test",\n        "checkpoint_loaded_allocated_baseline_bytes": 3096866304,\n        "idle_fp16_allocated_baseline_bytes": 3098177024,\n        "cleanup_reference_ledger": "allocation_executor_after_rebuilt_FP16_20261127T212546_418198Z.json",\n        "cleanup_accounting": {\n          "passed": true,\n          "problems": [],\n          "before_allocated_bytes": 3096866304,\n          "after_allocated_bytes": 3098177024,\n          "allocated_delta_bytes": 1310720,\n          "model_allocator_block_delta_bytes": 1310720,\n          "nonmodel_allocator_block_delta_bytes": 0,\n          "model_tensor_storage_bytes": 3087428864,\n          "before_ledger": "allocation_executor_before_checkpoint_loaded_FP16_20261127T212530_817152Z.json",\n          "after_ledger": "allocation_executor_after_rebuilt_FP16_20261127T212546_418198Z.json",\n          "rule": "same model storage inventory; no additional nonmodel active requests or pending backing; all block sizes reconciled",\n          "exact_allocated_byte_equality": false\n        },\n        "tracked_request_tensors_released": true,\n        "unique_tensor_backing_bytes": 3087428864,\n        "all_state_on_gpu_0": true,\n        "awq_layers": []\n      }\n    },\n    "idle": {\n      "status": "PASS",\n      "setup_or_stage_wall_seconds": 66.68929412,\n      "details": {\n        "layers": [\n          0,\n          1,\n          2,\n          3,\n          4,\n          5,\n          6,\n          7,\n          8,\n          9,\n          10,\n          11,\n          12,\n          13,\n          14,\n          15,\n          16,\n          17,\n          18,\n          19,\n          20,\n          21,\n          22,\n          23,\n          24,\n          25,\n          26,\n          27\n        ],\n        "directions": 2,\n        "measured_latency_samples_expected": 560,\n        "reference_states": "all FP16 -> exactly layer j AWQ -> all FP16; not a LIS path"\n      }\n    },\n    "contention": {\n      "status": "PASS",\n      "setup_or_stage_wall_seconds": 315.71560948,\n      "details": {\n        "expected_cases": 18,\n        "fixtures_not_LIS": true,\n        "shapes": [\n          {\n            "phase": "prefill",\n            "batch": 1,\n            "length": 256\n          },\n          {\n            "phase": "prefill",\n            "batch": 1,\n            "length": 1024\n          },\n          {\n            "phase": "decode",\n            "batch": 1,\n            "length": 512\n          }\n        ]\n      }\n    },\n    "serial_stall": {\n      "status": "PASS",\n      "setup_or_stage_wall_seconds": 107.68176760799997,\n      "details": {\n        "expected_cases": 18,\n        "fixtures_not_LIS": true,\n        "shapes": [\n          {\n            "phase": "prefill",\n            "batch": 1,\n            "length": 256\n          },\n          {\n            "phase": "prefill",\n            "batch": 1,\n            "length": 1024\n          },\n          {\n            "phase": "decode",\n            "batch": 1,\n            "length": 512\n          }\n        ]\n      }\n    },\n    "final_restoration": {\n      "status": "PASS",\n      "setup_or_stage_wall_seconds": 8.881853660999923,\n      "details": {\n        "every_original_decoder_tensor_exact": true,\n        "active_awq_layers": [],\n        "fresh_cache_fp16_max_absolute_difference": 0.0,\n        "allocated_baseline_bytes": 3096866304,\n        "cleanup_accounting": {\n          "passed": true,\n          "problems": [],\n          "before_allocated_bytes": 3098177024,\n          "after_allocated_bytes": 3096866304,\n          "allocated_delta_bytes": -1310720,\n          "model_allocator_block_delta_bytes": -1310720,\n          "nonmodel_allocator_block_delta_bytes": 0,\n          "model_tensor_storage_bytes": 3087428864,\n          "before_ledger": "allocation_executor_after_rebuilt_FP16_20261127T212546_418198Z.json",\n          "after_ledger": "allocation_final_restoration_FP16_20261127T213405_490602Z.json",\n          "rule": "same model storage inventory; no additional nonmodel active requests or pending backing; all block sizes reconciled",\n          "exact_allocated_byte_equality": false\n        },\n        "ownership": {\n          "unique_tensor_backing_bytes": 3087428864,\n          "all_state_on_gpu_0": true,\n          "awq_layers": []\n        },\n        "native_performance_callable_restored": true\n      }\n    }\n  },\n  "completion_status": "PASS_CONFIGURED_SCOPE",\n  "units": {\n    "time": "ms",\n    "memory": "bytes"\n  },\n  "scope": "Whole-decoder-layer reference costs; no page remapping, LIS path, or serving-overlap calibration",\n  "unmeasured": [\n    "elastic KV page detachment/remapping",\n    "aligned page capacity",\n    "asynchronous committed layer replacement",\n    "unaffected-layer hiding windows",\n    "semantic output quality",\n    "full burst-to-FP16 recovery time"\n  ],\n  "environment_signature": "9981971b5ed6546543e32b4a2fed6c1ad71855f041eb2fb56f33e6fbb080d85e",\n  "export_utc": "2026-11-27T21:34:05.795247+00:00",\n  "coverage": {\n    "expected_cases": 64,\n    "completed_cases": 64,\n    "missing_case_ids": [],\n    "unexpected_case_ids": [],\n    "errors": [],\n    "full_28_layer_idle_grid": true,\n    "pass": true\n  },\n  "executed_function_sources_sha256": "2570e7371788cb099f30b41b52bda5790afcf6b2af1702f5911031c0b9e30aab"\n}\n'
PRIOR_SWITCHING_ZIP_SHA256='c6d2a7cb3a84e18578687c60d994b963c36f4153c11698e1d806ae8be3fb45d3'
PROFILER_SOURCE_BASE64='...'
RESULT_README="# ReServe greedy LIS and intermediate profiles — schema 1\n\nThis is the step after the reviewed switching profiler. It produces a fixed 28-layer order,\nnative per-operation time/memory records, and a held-out timing-interpolation diagnostic.\nThe validated original FP16 and prepared AWQ checkpoints, whole-layer boundary (including both\nchanged norms), one active GPU model, pinned CPU sources and native T4 software stack are retained.\n\n## Run\n\nImport the delivered notebook into a fresh Kaggle notebook. Enable Internet and GPU T4. Only GPU 0 is used.\nRun Cell 1; if packages are installed, restart the kernel and run from the top. Run all later cells in order.\nNo prior ZIP upload is required: the exact compatibility bundle and reviewed switching provenance are embedded.\nWikiText train/validation data are downloaded from a pinned repository commit.\n\nDefault work includes 28 static local scores, 406 conditional greedy candidates, 29 configuration footprints,\nand 200 operation cases: 128 fitting, 64 validation, 8 endpoint anchor cases. Each operation case has\n3 warmups, 10 timing trials and 3 separate memory trials. This is substantially more work than the switching run.\nCompleted case attempts and greedy rounds are saved atomically. To resume after a kernel restart, keep the\nsame writable run folder, set RESUME_DIRECTORY to it and run from the top without changing settings/source.\nLarge activation scratch stays in scratch_cache and is excluded from the exported ZIP. If that scratch is lost\nbefore static scoring finishes, it is reconstructed from the frozen inputs. A different source/settings needs\na new output directory. Download partial results through the final cell if any stage fails.\n\n## Method\n\nThe score and greedy direction follow Methodology Section 3 and MorphServe Algorithm 1:\nLIS = 0.25 LTS + 0.25 LRS + 0.50 MDS. Choose the highest score; exact ties use ascending layer index.\nLTS compares original FP16 input/output at sampled nonpadding positions. LRS replays the complete original\nlayer input with its original attention/rotary arguments, then compares FP16/AWQ outputs at the same positions.\nOriginal full-context FP16 replay is checked before AWQ scoring. MDS compares final model logits before and\nafter adding each candidate to the already quantized prefix. MDS is recomputed every round; a one-time sort\nis not used. Reverse order restores the nested configuration path.\n\nThe initial calibration budget is our explicit ReServe choice: 8 non-overlapping 2,048-token WikiText windows\nand 32 fixed stratified token positions per example. It is not claimed to reproduce a paper-specific sample\ncount or demonstrate optimality/semantic quality. Cosines are per token in float64 CPU arithmetic, then\naveraged over all recorded positions/examples. Both zero vectors yield 1; one zero vector yields 0.\n\nScoring temporarily selects split-K 1 where GEMM is dispatched, with finally-based restoration. At the default\n2,048-token calibration length, AutoAWQ actually uses its large-token dequantize/matmul path; the split-K override\ndoes not alter that branch. Shorter calibration fixtures would use serial GEMM. Performance/memory profiling\nuses the native callable with split-K 8 on its GEMM branch. Profiling never uses serial-kernel timing.\n\nInputs are frozen before scoring: LIS uses the first half of the train token stream, timing fitting uses\nnon-overlapping windows from the second half, and held-out timing configurations use the separate validation\nsplit. The test split and future scheduler evaluation remain untouched. The order is frozen before any\nheld-out timing result is inspected. This is a timing-configuration holdout; the greedy algorithm necessarily\nvisits intermediate prefixes on its own separate calibration data.\n\nThe operation fit configurations are q=1,7,14,21; q=10,18 are held out from cost fitting. Each uses batch\n1,2,4,8 at lengths 256,512,1,024,2,048, separate full-prefill and one-step cached decode. Endpoints q=0,28\nreceive only targeted B=1, lengths 256/1,024 checks. The old complete fixed-mode profiles are not repeated.\nEvery decode trial starts with exactly the declared cached length; it advances the entire batch by one token.\nModel/tokenizer/backend, q, exact AWQ set/hash, data role, phase and shapes identify each cost. Times are ms;\nsizes are bytes. Do not divide decode batch duration by batch size in the simulator.\n\n## Outputs\n\n| File | Meaning |\n|---|---|\n| metadata.json | Source/config hashes, stage outcomes, stack, prior provenance and units |\n| frozen_inputs.json | Dataset commit/file hashes, source spans, token IDs, masks, positions and disjoint data roles |\n| layer_order.json | Sealed permutation, reverse restoration order, greedy margins and all prefix identities |\n| static_scores.csv, static_token_scores.csv | LTS/LRS summaries and raw cosine samples |\n| greedy_scores.csv, greedy_token_scores.csv | Every conditional candidate score and its raw MDS cosines |\n| configuration_footprints.json/.csv | Tensor backing, actual allocator blocks, allocated/reserved baselines for all 29 q values |\n| operation_trials.csv, operation_summary.csv | Raw time/peak-memory observations and summaries; warmups/probes excluded from latency stats |\n| operation_memory_snapshots.csv | Instrumented lifecycle samples, including prepared decode cache and request release |\n| interpolation_validation.json, interpolation_residuals.csv | Frozen-q interpolation residuals on held-out configs/inputs |\n| path_validation.json, final_restoration.json | Native retained-KV path continuity, exact weights and original fresh FP16 logits |\n| allocation_*.json | Detailed allocator ledgers; tensor sizes and allocator block padding stay distinct |\n| cases/ | Complete and failed/interrupted attempts, each with raw records |\n| errors.json, coverage_audit.json, file_hashes.json | Coverage, current failures and file integrity |\n| compatibility_input.zip, prior_switching_metadata.json | Original compatibility evidence and reviewed switching provenance |\n| distributed_notebook_source.py, executed_function_sources.py, reproduce.py | Distributed implementation, captured helper definitions and standalone rerun entry point |\n\nThe native operation GPU-event duration and synchronized host duration are different views; do not sum them.\nModel preparation/swapping, input transfer, decode-cache prefill setup, validation and file writes occur outside\nthe measured operation. Decode operation peaks are reset after its prefill setup, so setup peaks are not\nmislabeled as decode workspace; instrumented snapshots preserve setup separately. Peaks include weights,\nrequest tensors, KV, outputs and execution temporaries; no isolated-workspace or page-arena inference is made.\nEach helper returns only CPU records and weakrefs. Request-state release is checked after the helper returns.\nAllocator padding differences are explained by ledgers rather than hidden in a loose byte tolerance.\n\n## Interpolation scope and outcome\n\nInterpolation is linear only in q, between adjacent fitting configurations at identical phase/batch/context\nand kernel dispatch. No extrapolation or batch/context interpolation is performed. The diagnostic separately\nreports GPU and host median-duration error. Limits are predeclared operation-level engineering checks:\nWAPE <=10% and every held-out case relative error <=20%. These are not the methodology's end-to-end P95\nTTFT/latency acceptance tests. A failed interpolation diagnostic preserves successful measurements and marks\nthe result COMPLETE_INTERPOLATION_NEEDS_REFINEMENT. Keep exact costs and densify the affected region before\nusing an unsupported approximation. Memory safety cannot be established by timing interpolation.\n\nCOMPLETE_INTERPOLATION_WITHIN_TESTED_LIMITS means all configured checks/cases finished and these operation\nlimits passed at the two tested held-out q values. It does not establish accuracy at every unmeasured q,\nunseen batch/context sizes, mixed batches, a CPU replay, semantic output quality, an elastic allocator,\nasynchronous layer overlap, or policy gains. The next work is reviewing this ZIP, freezing capacity and\ntransition accounting, and validating the profile-backed simulator before admission-policy comparisons.\n\nThis notebook was checked with 11 new CPU protocol/decision fixtures. Shared executor helpers come from the\nreviewed T4 switching implementation. The new notebook has not been executed on CUDA by its author in this\nsession. CPU fixtures do not validate new T4 measurements; submit the resulting ZIP for review.\n\nFor a standalone GPU rerun, keep extracted output files together and run `python reproduce.py` on the matched\nstack. If the environment installer runs, run the command again after it finishes. Stored run settings are\nreused, and a new output directory is created. If editing notebook helper source, retain that notebook version\nalongside the ZIP because the distributed source represents the delivered implementation.\n\n## References\n\n- Project methodology: https://chatgpt.com/space/page_da3e562851808191b47406c40260021f\n- MorphServe §4.3 and Appendix A/Algorithm 1: https://proceedings.mlsys.org/paper_files/paper/2026/file/8144a9d62e506af0fcdeac0e456b2710-Paper-Conference.pdf\n- WikiText: https://huggingface.co/datasets/Salesforce/wikitext/tree/b08601e04326c79dfdd32d625aee71d232d685c3/wikitext-2-raw-v1\n- Qwen2 4.51.3 selected-logit API: https://huggingface.co/docs/transformers/v4.51.3/en/model_doc/qwen2\n- PyTorch 2.6 CUDA timing/allocation: https://docs.pytorch.org/docs/2.6/notes/cuda.html\n"
REPRODUCE_SCRIPT='"""Run beside the extracted result files on the matching T4 stack."""\nfrom pathlib import Path\nimport base64,hashlib,json\nroot=Path(__file__).resolve().parent\nmetadata=json.loads((root/"metadata.json").read_text())\nsource=(root/"distributed_notebook_source.py").read_bytes()\nCODE_SHA256=metadata["code_sha256"]\nif hashlib.sha256(source).hexdigest()!=CODE_SHA256:\n    raise RuntimeError("Distributed source hash mismatch")\nPROFILER_SOURCE_BASE64=base64.b64encode(source).decode()\nCOMPATIBILITY_ZIP_BASE64=base64.b64encode((root/"compatibility_input.zip").read_bytes()).decode()\nPRIOR_SWITCHING_METADATA_JSON=(root/"prior_switching_metadata.json").read_text()\nPRIOR_SWITCHING_ZIP_SHA256=metadata["prior_switching_zip_sha256"]\nRESULT_README=(root/"README.md").read_text()\nREPRODUCE_SCRIPT=Path(__file__).read_text()\nREPLAY_SETTINGS=metadata["settings"]\nexec(compile(source.decode(),str(root/"distributed_notebook_source.py"),"exec"))\n'


In [ ]:
import base64
import copy
import csv
import gc
import hashlib
import inspect
import io
import json
import math
import os
import platform
import random
import shutil
import statistics
import subprocess
import time
import traceback
import weakref
import zipfile
from contextlib import contextmanager
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch
from huggingface_hub import snapshot_download, hf_hub_download
from safetensors import safe_open
from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM

RESUME_DIRECTORY = ""  # Empty for a new run. Set to an existing writable run folder after a kernel restart.
CFG = {
    "schema_version": 1, "seed": 20261127,
    "calibration_examples": 8, "calibration_length": 2048, "score_positions_per_example": 32,
    "score_weights": {"lts": 0.25, "lrs": 0.25, "mds": 0.50},
    "cosine_epsilon": 1e-12,
    "scoring_kernel": "Triton_serial_split_k_1_numerical_calibration_only",
    "performance_kernel": "Triton_native_split_k_8_SDPA",
    "fit_q": [1, 7, 14, 21], "validation_q": [10, 18],
    "batches": [1, 2, 4, 8], "lengths": [256, 512, 1024, 2048],
    "anchor_shapes": [[1, 256], [1, 1024]],
    "warmups": 3, "trials": 10, "memory_trials": 3,
    "operation_interpolation_wape_limit_percent": 10.0,
    "operation_interpolation_max_case_error_percent": 20.0,
    "allocation_layout": "whole_layer_allocate_copy_commit_release",
    "host_policy": "all_variants_pinned_no_silent_fallback",
    "dataset": {"repo": "Salesforce/wikitext", "revision": "b08601e04326c79dfdd32d625aee71d232d685c3",
                "subset": "wikitext-2-raw-v1"},
}
if "REPLAY_SETTINGS" in globals():
    CFG = copy.deepcopy(globals()["REPLAY_SETTINGS"])
SPECS = {
    "fp16": {"repo": "Qwen/Qwen2.5-1.5B-Instruct", "revision": "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"},
    "awq": {"repo": "Qwen/Qwen2.5-1.5B-Instruct-AWQ", "revision": "c5b72040769bfde6c5b562ae890f9657b0803f79"},
}
DEVICE = torch.device("cuda:0")
STAGE_ORDER = ["environment", "evidence", "host_setup", "data", "bootstrap",
               "static_scores", "greedy_order", "path_validation", "footprints",
               "profiles", "interpolation", "final_restoration"]

def require(ok, message):
    if not ok:
        raise RuntimeError(message)

def canonical_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False).encode()).hexdigest()

def config_identity(awq_layers):
    layers = sorted(awq_layers)
    return {"awq_layers": layers, "configuration_sha256": canonical_hash(layers)}

def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    with temporary.open("w") as f:
        json.dump(value, f, indent=2, allow_nan=False)
        f.write("\n")
        f.flush()
        os.fsync(f.fileno())
    os.replace(temporary, path)

def atomic_torch(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    torch.save(value, temporary)
    os.replace(temporary, path)

def save_csv(path, rows):
    path = Path(path)
    if not rows:
        path.write_text("")
        return
    fields = sorted({k for r in rows for k in r})
    with path.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        for row in rows:
            writer.writerow({k: json.dumps(v, sort_keys=True) if isinstance(v, (dict, list)) else v for k, v in row.items()})

SESSION_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
WORKING = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
OUT = Path(RESUME_DIRECTORY) if RESUME_DIRECTORY else WORKING / f"lis_intermediate_{SESSION_ID}"
OUT.mkdir(parents=True, exist_ok=bool(RESUME_DIRECTORY))
CACHE = OUT / "scratch_cache"
CACHE.mkdir(exist_ok=True)
require(1 <= CFG["score_positions_per_example"] <= CFG["calibration_length"], "Invalid scored token positions")
require(CFG["calibration_examples"] >= 2 and CFG["calibration_length"] >= 128, "Invalid calibration budget")
require(math.isclose(sum(CFG["score_weights"].values()), 1.0), "Invalid LIS weights")
require(CFG["score_weights"] == {"lts":.25,"lrs":.25,"mds":.5}, "Keep the methodology's frozen LIS weights")
require(not(set(CFG["fit_q"]) & set(CFG["validation_q"])), "Fitting and validation configurations overlap")
require(all(0 < q < 28 for q in CFG["fit_q"] + CFG["validation_q"]), "Invalid intermediate q")
require(min(CFG["fit_q"]) < min(CFG["validation_q"]) and max(CFG["validation_q"]) < max(CFG["fit_q"]), "Validation must interpolate inside the fitting range")
require(CFG["warmups"] >= 1 and CFG["trials"] >= 2 and CFG["memory_trials"] >= 1, "Invalid repetitions")
require(CFG["batches"] and CFG["lengths"] and all(b in [1,2,4,8] for b in CFG["batches"])
        and all(n in [256,512,1024,2048] for n in CFG["lengths"]), "Unsupported operation grid")
RUN_CONFIG = {"settings": CFG, "models": SPECS, "code_sha256": CODE_SHA256,
              "prior_switching_zip_sha256": PRIOR_SWITCHING_ZIP_SHA256}
CONFIG_HASH = canonical_hash(RUN_CONFIG)
if RESUME_DIRECTORY:
    REPORT = json.loads((OUT / "metadata.json").read_text())
    require(REPORT["configuration_sha256"] == CONFIG_HASH, "Configuration/source changed; use a fresh output folder")
    REPORT["checks"] = {s:{"status":"NOT_RUN"} for s in STAGE_ORDER}
else:
    REPORT = {"schema_version":1, "run_id":SESSION_ID, "configuration_sha256":CONFIG_HASH,
              **RUN_CONFIG, "sessions":[], "checks":{s:{"status":"NOT_RUN"} for s in STAGE_ORDER}}
REPORT["sessions"].append({"session_id":SESSION_ID,"start_utc":datetime.now(timezone.utc).isoformat()})
REPORT.update({"completion_status":"INCOMPLETE", "units":{"time":"ms","memory":"bytes"},
    "scope":"Frozen greedy LIS and homogeneous operation profiles on one T4; no elastic allocator or policy result",
    "unmeasured":["aligned KV pages and detachment/remapping", "mixed/heterogeneous batching",
                  "interpolation across unseen batch/context sizes", "end-to-end simulator replay",
                  "semantic output quality", "asynchronous committed layer overlap", "admission policy gains"]})

def save_report():
    atomic_json(OUT / "metadata.json", REPORT)

def archive_outputs():
    target = OUT.parent / (OUT.name + ".zip")
    with zipfile.ZipFile(target, "w", zipfile.ZIP_DEFLATED) as z:
        for p in sorted(OUT.rglob("*")):
            if p.is_file() and not p.name.endswith(".tmp") and not p.is_relative_to(CACHE):
                z.write(p, str(p.relative_to(OUT)))
    return target

def run_stage(name, fn):
    index = STAGE_ORDER.index(name)
    require(all(REPORT["checks"][s]["status"]=="PASS" for s in STAGE_ORDER[:index]), "Run preceding cells successfully first")
    for later in STAGE_ORDER[index:]:
        REPORT["checks"][later] = {"status":"NOT_RUN"}
    REPORT["checks"][name] = {"status":"RUNNING"}
    REPORT["completion_status"] = "INCOMPLETE"
    save_report()
    started = time.perf_counter()
    try:
        result = fn()
        REPORT["checks"][name] = {"status":"PASS", "stage_wall_seconds":time.perf_counter()-started, "details":result}
        save_report()
        print(name, "PASS", flush=True)
        return result
    except BaseException as exc:
        REPORT["checks"][name] = {"status":"INTERRUPTED" if isinstance(exc,KeyboardInterrupt) else "FAIL", "error":str(exc)}
        REPORT["completion_status"] = "INCOMPLETE_OR_FAILED"
        (OUT/f"{name}_{SESSION_ID}_error.txt").write_text(traceback.format_exc())
        save_report()
        try:
            archive_outputs()
        except Exception as archive_error:
            print("Partial archive error:", archive_error)
        print("Completed records remain saved. Run the final export cell for a partial ZIP. Restart before retrying GPU failures.")
        raise

def passed_case(case_id):
    for p in sorted((OUT/"cases").glob("*.json")):
        data = json.loads(p.read_text())
        if data["case_id"]==case_id and data["status"]=="PASS":
            return data
    return None

def new_case(case_id, kind, **extra):
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    case = {"case_id":case_id,"kind":kind,"status":"RUNNING","attempt":stamp,"session_id":SESSION_ID,
            "configuration_sha256":CONFIG_HASH,"rows":[],**extra}
    case["path"] = str(OUT/"cases"/f"{case_id}_{stamp}.json")
    checkpoint(case)
    return case

def checkpoint(case):
    atomic_json(case["path"],case)

def finish_case(case, fn):
    try:
        fn(case)
        case["status"] = "PASS"
    except BaseException as exc:
        case["status"] = "INTERRUPTED" if isinstance(exc,KeyboardInterrupt) else "FAIL"
        case["error"] = str(exc)
        raise
    finally:
        checkpoint(case)

save_report()
print("Output folder:",OUT)


def sampled_positions(length, count):
    require(1 <= count <= length, "Invalid position sample")
    if count == 1:
        return [length-1]
    return [i*(length-1)//(count-1) for i in range(count)]

def cosine_tokens(a, b, epsilon=None):
    epsilon = CFG["cosine_epsilon"] if epsilon is None else epsilon
    a, b = np.asarray(a,dtype=np.float64), np.asarray(b,dtype=np.float64)
    require(a.shape==b.shape and a.ndim>=2, "Cosine inputs must have matching token/vector dimensions")
    a,b = a.reshape(-1,a.shape[-1]),b.reshape(-1,b.shape[-1])
    require(len(a)>0 and np.isfinite(a).all() and np.isfinite(b).all(), "Nonfinite/empty score inputs")
    dots = np.einsum("ij,ij->i",a,b)
    an = np.sqrt(np.einsum("ij,ij->i",a,a))
    bn = np.sqrt(np.einsum("ij,ij->i",b,b))
    valid = (an>epsilon)&(bn>epsilon)
    values = np.zeros(len(a),dtype=np.float64)
    values[valid] = dots[valid]/(an[valid]*bn[valid])
    values[(an<=epsilon)&(bn<=epsilon)] = 1.0
    return np.clip(values,-1.0,1.0).tolist()

def lis_score(lts, lrs, mds, weights=None):
    w = CFG["score_weights"] if weights is None else weights
    return w["lts"]*lts+w["lrs"]*lrs+w["mds"]*mds

def greedy_winner(candidates):
    require(candidates and all(math.isfinite(r["lis"]) for r in candidates), "Invalid candidate scores")
    return min(candidates,key=lambda r:(-r["lis"],r["layer"]))

def adjacent_bracket(q, fitting_q):
    lower = max((x for x in fitting_q if x<q),default=None)
    upper = min((x for x in fitting_q if x>q),default=None)
    require(lower is not None and upper is not None, "Interpolation cannot extrapolate")
    return lower,upper

def linear_q_prediction(q, lo, hi, low_value, high_value):
    require(lo<q<hi, "Invalid interpolation bracket")
    return low_value+(high_value-low_value)*(q-lo)/(hi-lo)

def quantile(values, p):
    ordered = sorted(values)
    require(ordered, "Empty quantile")
    index = (len(ordered)-1)*p
    low,high = math.floor(index),math.ceil(index)
    return ordered[low]+(ordered[high]-ordered[low])*(index-low)


def profile_summary_rows(cases):
    result = []
    for case in cases:
        timing = [r for r in case["rows"] if not r["warmup"] and r["instrumentation"]=="timing"]
        probes = [r for r in case["rows"] if not r["warmup"] and r["instrumentation"]=="memory"]
        require(len(timing)==CFG["trials"] and len(probes)==CFG["memory_trials"], "Incomplete profile repetitions")
        record = {"case_id":case["case_id"],**case["spec"],
            "configuration_sha256":timing[0]["configuration_sha256"],"layer_order_sha256":case["layer_order_sha256"],
            "warmups":sum(r["warmup"] for r in case["rows"]),"trials":len(timing),"memory_trials":len(probes)}
        for name in ["gpu_operation_ms","synchronized_host_operation_ms"]:
            values = [r[name] for r in timing]
            record.update({name+"_median":statistics.median(values),name+"_mean":statistics.mean(values),
                name+"_std":statistics.stdev(values),name+"_p95":quantile(values,.95),name+"_min":min(values),name+"_max":max(values)})
        record.update({"observed_peak_allocated_bytes_max":max(r["operation_peak_allocated_bytes"] for r in timing+probes),
            "observed_peak_reserved_bytes_max":max(r["operation_peak_reserved_bytes"] for r in timing+probes),
            "raw_KV_bytes_after":timing[0]["logical_KV_bytes"],"AWQ_dispatch":timing[0]["AWQ_dispatch"],
            "cleanup_verified":case["cleanup_accounting"]["passed"]})
        result.append(record)
    return result


def interpolation_rows(summaries, fitting_q):
    lookup = {(s["role"],s["q"],s["phase"],s["batch"],s["length"]):s for s in summaries}
    result = []
    for measured in summaries:
        if measured["role"]!="validation":
            continue
        q = measured["q"]
        lo,hi = adjacent_bracket(q,fitting_q)
        low = lookup[("fit",lo,measured["phase"],measured["batch"],measured["length"])]
        high = lookup[("fit",hi,measured["phase"],measured["batch"],measured["length"])]
        require(low["AWQ_dispatch"]==high["AWQ_dispatch"]==measured["AWQ_dispatch"], "Interpolation crossed a kernel dispatch boundary")
        for metric in ["gpu_operation_ms_median","synchronized_host_operation_ms_median"]:
            predicted = linear_q_prediction(q,lo,hi,low[metric],high[metric])
            actual = measured[metric]
            require(actual>0 and predicted>0, "Invalid timing in interpolation")
            result.append({"q":q,"phase":measured["phase"],"batch":measured["batch"],"length":measured["length"],
                "metric":metric,"lower_q":lo,"upper_q":hi,"predicted_ms":predicted,"measured_ms":actual,
                "absolute_error_ms":abs(predicted-actual),"absolute_relative_error_percent":100*abs(predicted-actual)/actual,
                "AWQ_dispatch":measured["AWQ_dispatch"],"held_out_configuration_and_input_bank":True})
    return result


## 3. Check GPU 0 and the native profiling backend

T4, CUDA 12.4, SDPA, unfused AutoAWQ GEMM/Triton and the previous package versions are checked.
The actual dataset, source, allocator and kernel settings are recorded; mismatched runs are not pooled.


In [ ]:
MODEL = None
HOST = {}
ACTIVE_AWQ = set()
COPY_STREAM = None
COMPUTE_STREAM = None
GEMM = None
NATIVE_GEMM = None
ORIGINAL_EXTENSION = None

def environment_check():
    global COPY_STREAM, COMPUTE_STREAM, GEMM, NATIVE_GEMM, ORIGINAL_EXTENSION
    require("PINS" in globals() and "installed_version" in globals(), "Run the installation cell first in this kernel")
    require(torch.cuda.is_available(), "Enable GPU T4 in Kaggle notebook settings")
    torch.cuda.set_device(0)
    require("T4" in torch.cuda.get_device_name(0), "Use one Tesla T4 to match the calibration")
    require(torch.version.cuda == "12.4", "CUDA runtime must be 12.4")
    require(os.environ.get("CUDA_LAUNCH_BLOCKING", "0") in ["0", ""], "Disable CUDA_LAUNCH_BLOCKING")
    require(torch.cuda.memory.get_allocator_backend() == "native", "Use PyTorch native allocator")
    for k, v in PINS.items():
        require((installed_version(k) or "").split("+")[0] == v, f"Restart after installing {k}=={v}")
    import awq.modules.linear.gemm as gemm
    GEMM = gemm
    require(gemm.TRITON_AVAILABLE, "AutoAWQ Triton unavailable")
    ORIGINAL_EXTENSION = gemm.awq_ext
    gemm.awq_ext = None
    NATIVE_GEMM = gemm.awq_gemm_triton
    source = inspect.getsource(gemm.WQLinearMMFunction.forward)
    require("split_k_iters=8" in source, "Unexpected native GEMM dispatch")
    require("1024" in source, "Unexpected large-token dispatch boundary")
    torch.manual_seed(CFG["seed"])
    torch.cuda.manual_seed_all(CFG["seed"])
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    COPY_STREAM = torch.cuda.Stream(device=0)
    COMPUTE_STREAM = torch.cuda.default_stream(0)
    props = torch.cuda.get_device_properties(0)
    environment = {"python": platform.python_version(), "packages": {k: installed_version(k) for k in PINS},
        "torch": torch.__version__, "cuda_runtime": torch.version.cuda, "gpu": props.name,
        "gpu_total_bytes": props.total_memory, "visible_gpus": torch.cuda.device_count(), "selected_gpu": 0,
        "allocator_backend": torch.cuda.memory.get_allocator_backend(),
        "allocator_env": {k: os.environ.get(k) for k in ["PYTORCH_CUDA_ALLOC_CONF", "CUDA_VISIBLE_DEVICES"]},
        "attention": "sdpa", "fused_modules": False, "split_k": 8,
        "optional_extension_present_before_override": ORIGINAL_EXTENSION is not None,
        "gemm_forward_source_sha256": hashlib.sha256(source.encode()).hexdigest(),
        "copy_stream_id": COPY_STREAM.cuda_stream, "compute_stream_id": COMPUTE_STREAM.cuda_stream}
    comparable = {k: environment[k] for k in ["packages", "cuda_runtime", "gpu", "gpu_total_bytes",
                                            "allocator_backend", "allocator_env", "gemm_forward_source_sha256"]}
    if REPORT.get("environment_signature"):
        require(REPORT["environment_signature"] == canonical_hash(comparable), "Resume environment changed")
    REPORT["environment_signature"] = canonical_hash(comparable)
    REPORT["sessions"][-1]["environment"] = environment
    try:
        (OUT / f"nvidia_smi_{SESSION_ID}.txt").write_text(subprocess.check_output(
            ["nvidia-smi", "-i", "0", "-q"], text=True))
    except Exception as exc:
        environment["nvidia_smi_error"] = str(exc)
    return environment

run_stage("environment", environment_check)


## 4. Read the passed compatibility evidence and pinned checkpoint snapshots

The exact compatibility ZIP SHA and final PASS scope are checked. The reviewed switching provenance is
saved as a separate input record. Existing checkpoint downloads are reused from Kaggle's cache.


In [ ]:
EXPECTED_COMPATIBILITY_SHA256 = "46f0f521cfec25c165f56a5f5faeae31550940b9a9a107440f3da3f7bf181649"
MANIFEST = None
PATHS = {}
MODEL_CONFIG = None
TOKENIZER = None

def evidence_check():
    global MANIFEST, PATHS, MODEL_CONFIG, TOKENIZER
    raw = base64.b64decode(COMPATIBILITY_ZIP_BASE64, validate=True)
    require(hashlib.sha256(raw).hexdigest() == EXPECTED_COMPATIBILITY_SHA256, "Embedded evidence SHA mismatch")
    require(len(raw) == 203994, "Unexpected compatibility ZIP length")
    evidence_dir = OUT / "compatibility_evidence"
    with zipfile.ZipFile(io.BytesIO(raw)) as z:
        report = json.loads(z.read("compatibility_report.json"))
        require(report["verdict"] == "PASS" and report["completion_status"] == "PASS", "Compatibility gate not complete")
        require(report["compatibility_execution"]["performance_profiling_split_k"] == 8, "Wrong performance kernel")
        for mode, spec in SPECS.items():
            require(report["models"][mode]["repo"] == spec["repo"] and
                    report["models"][mode]["resolved_revision"] == spec["revision"], "Checkpoint identity mismatch")
        MANIFEST = json.loads(z.read("variant_manifests.json"))
        require(len(MANIFEST["layers"]) == 28 and not MANIFEST["problems"], "Invalid variant manifests")
        for j, layer in enumerate(MANIFEST["layers"]):
            require(layer["layer"] == j and len(layer["changed_companions"]) == 2, "Layer/companion manifest mismatch")
            for mode in SPECS:
                variant = layer["variants"][mode]
                require(sum(t["bytes"] for t in variant["all_layer_tensors"]) == variant["full_layer_bytes"], "Payload accounting mismatch")
        for item in z.infolist():
            target = evidence_dir / item.filename
            require(target.resolve().is_relative_to(evidence_dir.resolve()), "Unsafe evidence path")
            if not item.is_dir():
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(z.read(item))
    (OUT / "compatibility_input.zip").write_bytes(raw)
    for mode, spec in SPECS.items():
        PATHS[mode] = Path(snapshot_download(repo_id=spec["repo"], revision=spec["revision"],
            allow_patterns=["*.json", "*.safetensors", "*.txt", "*.model", "*.jinja"]))
        require(PATHS[mode].name == spec["revision"], "Snapshot did not resolve to pinned revision")
    MODEL_CONFIG = AutoConfig.from_pretrained(PATHS["fp16"], local_files_only=True, trust_remote_code=False)
    MODEL_CONFIG._attn_implementation = "sdpa"
    require(MODEL_CONFIG.num_hidden_layers == 28, "Unexpected layer count")
    TOKENIZER = AutoTokenizer.from_pretrained(PATHS["fp16"], local_files_only=True, trust_remote_code=False)
    return {"compatibility_sha256": EXPECTED_COMPATIBILITY_SHA256,
            "variant_manifest_sha256": hashlib.sha256((evidence_dir / "variant_manifests.json").read_bytes()).hexdigest(),
            "boundary": "complete decoder module including both companion norms",
            "tokenizer_and_chat_template": "pinned FP16 tokenizer for every configuration"}

def evidence_with_prior():
    result = evidence_check()
    prior = json.loads(PRIOR_SWITCHING_METADATA_JSON)
    require(prior["completion_status"] == "PASS_CONFIGURED_SCOPE", "Prior switching run did not pass")
    require(prior["code_sha256"] == "3f3dec822e48b55346dcc1ce2a04d66db4c2ba91de24b16a0d07bf2c12ed57f9", "Unexpected reviewed switching source")
    require(all(v["status"] == "PASS" for v in prior["checks"].values()), "Prior run has incomplete stages")
    previous_env = prior["sessions"][-1]["environment"]
    for name, version in previous_env["packages"].items():
        require(installed_version(name) == version, f"Current {name} differs from the reviewed switching run")
    require(platform.python_version() == previous_env["python"], "Python version differs from reviewed calibration")
    atomic_json(OUT / "prior_switching_metadata.json", prior)
    result["reviewed_switching_zip_sha256"] = PRIOR_SWITCHING_ZIP_SHA256
    result["prior_switching_results_embedded"] = "metadata/provenance only; original raw ZIP remains the authoritative prior artifact"
    return result

run_stage("evidence", evidence_with_prior)


## 5. Stage the original FP16/AWQ variants and define the checked executor

Both complete decoder variants remain in pinned CPU memory, about 3.08 GiB of tensor backing plus overhead.
One GPU model is loaded. Complete-layer swaps include both changed normalization vectors; restoration uses
the original FP16 tensors. The following helpers are reused from the successful switching implementation.
No switching-cost grid is rerun.


In [ ]:
def tensor_digest(tensor):
    value = tensor.detach().cpu().contiguous().reshape(-1)
    return hashlib.sha256(value.view(torch.uint8).numpy().tobytes()).hexdigest()

def host_observation():
    values = {}
    if Path("/proc/meminfo").exists():
        for line in Path("/proc/meminfo").read_text().splitlines():
            key, rest = line.split(":", 1)
            if key in ["MemTotal", "MemAvailable", "Mlocked", "Unevictable"]:
                values[key + "_bytes"] = int(rest.strip().split()[0]) * 1024
    if Path("/proc/self/status").exists():
        for line in Path("/proc/self/status").read_text().splitlines():
            key, rest = line.split(":", 1)
            if key in ["VmRSS", "VmLck", "VmPin"]:
                values[key + "_bytes"] = int(rest.strip().split()[0]) * 1024
    return values

def host_setup():
    global MODEL, HOST
    require(not HOST and MODEL is None, "Host setup already exists; restart to repeat this cell")
    expected_bytes = sum(x["variants"][mode]["full_layer_bytes"] for x in MANIFEST["layers"] for mode in SPECS)
    host_report = {"policy": CFG["host_policy"], "expected_decoder_backup_bytes": expected_bytes,
        "scope": "Both complete decoder variants; global tensors stay resident and are not switched",
        "before": host_observation(), "layers": [], "status": "RUNNING"}
    atomic_json(OUT / "host_memory.json", host_report)
    try:
        available = host_report["before"].get("MemAvailable_bytes")
        if available is not None:
            require(available > expected_bytes + 2 * 1024**3, "Insufficient host RAM for all pinned variants plus setup")
        for mode in SPECS:
            index = PATHS[mode] / "model.safetensors.index.json"
            if index.exists():
                mapping = json.loads(index.read_text())["weight_map"]
            else:
                with safe_open(str(PATHS[mode] / "model.safetensors"), framework="pt", device="cpu") as f:
                    mapping = {k: "model.safetensors" for k in f.keys()}
            for j, item in enumerate(MANIFEST["layers"]):
                state = {}
                start = time.perf_counter()
                pin_ms = 0.0
                prefix = f"model.layers.{j}."
                for entry in item["variants"][mode]["all_layer_tensors"]:
                    name = entry["name"]
                    with safe_open(str(PATHS[mode] / mapping[name]), framework="pt", device="cpu") as f:
                        original = f.get_tensor(name)
                    converted = original.to(torch.float16) if original.is_floating_point() else original
                    pin_start = time.perf_counter_ns()
                    pinned = converted.contiguous().pin_memory()
                    pin_ms += (time.perf_counter_ns()-pin_start)/1e6
                    require(pinned.is_pinned() and pinned.device.type == "cpu", "Host pinning did not succeed")
                    require(list(pinned.shape) == entry["shape"] and pinned.numel()*pinned.element_size() == entry["bytes"], "Host tensor shape/bytes mismatch")
                    require(tensor_digest(pinned) == entry["normalized_sha256"], f"Host source differs: {name}")
                    state[name[len(prefix):]] = pinned
                    del original, converted, pinned
                HOST[(mode, j)] = state
                host_report["layers"].append({"mode": mode, "layer": j,
                    "bytes": sum(t.numel()*t.element_size() for t in state.values()),
                    "pinning_host_ms": pin_ms,
                    "load_convert_pin_hash_wall_ms": 1000*(time.perf_counter()-start)})
                atomic_json(OUT / "host_memory.json", host_report)
            print(mode, "all 28 source variants pinned and hashed", flush=True)
        actual = sum(t.numel()*t.element_size() for state in HOST.values() for t in state.values())
        require(actual == expected_bytes, "Pinned backing byte accounting mismatch")
        host_report.update({"actual_pinned_tensor_bytes": actual, "all_required_variants_pinned": True,
                            "after_pinning": host_observation()})
        # Only one complete GPU model exists. Restore always uses HOST original values.
        MODEL, info = AutoModelForCausalLM.from_pretrained(PATHS["fp16"], torch_dtype=torch.float16,
            attn_implementation="sdpa", local_files_only=True, trust_remote_code=False,
            use_safetensors=True, low_cpu_mem_usage=True, output_loading_info=True)
        allowed = {"lm_head.weight", "model.embed_tokens.weight"} if MODEL_CONFIG.tie_word_embeddings else set()
        require(not(set(info.get("missing_keys", []))-allowed) and not any(info.get(k) for k in
            ["unexpected_keys", "mismatched_keys", "error_msgs"]), f"Loading mismatch: {info}")
        MODEL.to(DEVICE).eval().requires_grad_(False)
        from accelerate.hooks import remove_hook_from_module
        remove_hook_from_module(MODEL, recurse=True)
        torch.cuda.synchronize(DEVICE)
        host_report.update({"status": "PASS", "after_gpu_loading": host_observation(),
                            "checkpoint_disk_bytes": {m: sum(p.stat().st_size for p in PATHS[m].glob("*.safetensors")) for m in SPECS}})
        return {"pinned_tensor_bytes": actual, "complete_variant_sets": 2, "gpu_models": 1}
    except BaseException as exc:
        host_report.update({"status": "FAIL", "error": str(exc),
            "pinned_bytes_before_failure": sum(t.numel()*t.element_size() for s in HOST.values() for t in s.values()),
            "fallback": "none; partial results saved; restart rather than label pageable transfers pinned"})
        raise
    finally:
        host_report["last_observation"] = host_observation()
        atomic_json(OUT / "host_memory.json", host_report)

run_stage("host_setup", host_setup)


In [ ]:
def build_allocation_ledger(segments, storages, allocated_bytes, label, device_index=0):
    """Pure accounting: tensor storage capacity and allocator block sizes are distinct."""
    blocks = []
    for segment in segments:
        if segment.get("device", device_index) != device_index:
            continue
        cursor = segment["address"]
        for block in segment["blocks"]:
            address = block.get("address", cursor)
            cursor = address + block["size"]
            if block["state"] in ["active_allocated", "active_awaiting_free"]:
                blocks.append({"address": address, "size": block["size"],
                    "requested_size": block.get("requested_size"), "state": block["state"],
                    "stream": segment.get("stream"), "model_storage_addresses": []})
    active = [b for b in blocks if b["state"] == "active_allocated"]
    problems = []
    for storage in storages:
        matching = [b for b in active if b["address"] <= storage["address"]
                    and storage["address"] + storage["bytes"] <= b["address"] + b["size"]]
        if len(matching) != 1:
            problems.append({"storage_not_in_one_active_block": storage})
        else:
            matching[0]["model_storage_addresses"].append(storage["address"])
    model = [b for b in active if b["model_storage_addresses"]]
    other = [b for b in active if not b["model_storage_addresses"]]
    if any(not isinstance(b["requested_size"], int) or b["requested_size"] <= 0 for b in active):
        problems.append("An active block lacks a positive requested_size")
    if sum(b["size"] for b in active) != allocated_bytes:
        problems.append("Snapshot active block sum differs from memory_allocated")
    logical = sorted([{"bytes": s["bytes"], "tensors": sorted(s["tensors"], key=lambda x: x["name"])}
                      for s in storages], key=lambda x: x["tensors"][0]["name"])
    return {"label": label, "valid": not problems, "problems": problems,
        "allocated_bytes": allocated_bytes,
        "model_tensor_storage_bytes": sum(s["bytes"] for s in storages),
        "model_storage_signature": canonical_hash(logical),
        "model_allocator_block_bytes": sum(b["size"] for b in model),
        "nonmodel_allocator_block_bytes": sum(b["size"] for b in other),
        "nonmodel_requested_sizes": sorted(b["requested_size"] for b in other if isinstance(b["requested_size"], int)),
        "pending_free_block_bytes": sum(b["size"] for b in blocks if b["state"] == "active_awaiting_free"),
        "active_blocks": blocks, "model_storages": storages}

def allocation_ledger(label, module=None):
    # Only call outside performance timing, or inside a labeled memory probe.
    module = MODEL if module is None else module
    torch.cuda.synchronize(DEVICE)
    gc.collect()
    storages = {}
    tensors = list(module.named_parameters(remove_duplicate=False)) + list(module.named_buffers(remove_duplicate=False))
    for name, t in tensors:
        if t.device != DEVICE or not t.numel():
            continue
        storage = t.untyped_storage()
        address, size = storage.data_ptr(), storage.nbytes()
        entry = storages.setdefault(address, {"address": address, "bytes": size, "tensors": []})
        require(entry["bytes"] == size, "Inconsistent storage capacity")
        entry["tensors"].append({"name": name, "shape": list(t.shape), "dtype": str(t.dtype)})
    snapshot = torch.cuda.memory_snapshot()
    ledger = build_allocation_ledger(snapshot, list(storages.values()),
                                    torch.cuda.memory_allocated(DEVICE), label)
    safe_label = "".join(c if c.isalnum() or c in "_-" else "_" for c in label)
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    ledger["report_file"] = f"allocation_{safe_label}_{stamp}.json"
    atomic_json(OUT / ledger["report_file"], ledger)
    return ledger

def compare_cleanup_ledgers(before, after):
    """Reject new nonmodel live allocations; explain every allocated-byte delta."""
    problems = []
    if not before["valid"] or not after["valid"]:
        problems.append("Invalid allocation ledger; inspect its problems")
    if before["model_storage_signature"] != after["model_storage_signature"]:
        problems.append("Model tensor names/shapes/dtypes/storage capacities or aliases changed")
    if before["model_tensor_storage_bytes"] != after["model_tensor_storage_bytes"]:
        problems.append("Model tensor backing changed")
    allowed = {}
    for size in before["nonmodel_requested_sizes"]:
        allowed[size] = allowed.get(size, 0) + 1
    for size in after["nonmodel_requested_sizes"]:
        allowed[size] = allowed.get(size, 0) - 1
    new_requests = {str(k): -v for k, v in allowed.items() if v < 0}
    if new_requests:
        problems.append({"unexplained_new_nonmodel_active_allocations_by_requested_size": new_requests})
    if after["pending_free_block_bytes"] > before["pending_free_block_bytes"]:
        problems.append("Additional backing remains pending release")
    delta = after["allocated_bytes"] - before["allocated_bytes"]
    model_delta = after["model_allocator_block_bytes"] - before["model_allocator_block_bytes"]
    other_delta = after["nonmodel_allocator_block_bytes"] - before["nonmodel_allocator_block_bytes"]
    if delta != model_delta + other_delta:
        problems.append("Allocated-byte difference is not reconciled by the active-block ledger")
    return {"passed": not problems, "problems": problems,
        "before_allocated_bytes": before["allocated_bytes"], "after_allocated_bytes": after["allocated_bytes"],
        "allocated_delta_bytes": delta, "model_allocator_block_delta_bytes": model_delta,
        "nonmodel_allocator_block_delta_bytes": other_delta,
        "model_tensor_storage_bytes": after["model_tensor_storage_bytes"],
        "before_ledger": before.get("report_file"), "after_ledger": after.get("report_file"),
        "rule": "same model storage inventory; no additional nonmodel active requests or pending backing; all block sizes reconciled",
        "exact_allocated_byte_equality": delta == 0}

def require_cleanup_ledgers(before, after, output_name):
    result = compare_cleanup_ledgers(before, after)
    atomic_json(OUT / output_name, result)
    require(result["passed"],
        f"Cleanup accounting failed: before={before['allocated_bytes']}, after={after['allocated_bytes']}, "
        f"delta={result['allocated_delta_bytes']} bytes. See {output_name} and its allocation ledgers. "
        f"Problems: {result['problems']}")
    return result


LINEARS = ["self_attn.q_proj", "self_attn.k_proj", "self_attn.v_proj", "self_attn.o_proj",
           "mlp.gate_proj", "mlp.up_proj", "mlp.down_proj"]
EXECUTOR_HEALTHY = True

def assert_native():
    require(GEMM.awq_gemm_triton is NATIVE_GEMM and GEMM.awq_ext is None,
            "Performance trials require native split-K 8 Triton; a temporary override remains active")

def skeleton(mode, j):
    from transformers.models.qwen2.modeling_qwen2 import Qwen2DecoderLayer
    from awq.modules.linear.gemm import WQLinear_GEMM
    with torch.device("meta"):
        layer = Qwen2DecoderLayer(copy.deepcopy(MODEL_CONFIG), layer_idx=j)
        if mode == "awq":
            for path in LINEARS:
                parent_path, attr = path.rsplit(".", 1)
                parent = layer.get_submodule(parent_path)
                dense = getattr(parent, attr)
                setattr(parent, attr, WQLinear_GEMM(w_bit=4, group_size=128,
                    in_features=dense.in_features, out_features=dense.out_features,
                    bias=dense.bias is not None, dev="meta"))
        layer.half().eval().requires_grad_(False)
    return layer

def variant_bytes(mode, j):
    return MANIFEST["layers"][j]["variants"][mode]["full_layer_bytes"]

def layer_bindings(layer, mode, j):
    state = layer.state_dict(keep_vars=True)
    source = HOST[(mode, j)]
    require(set(state) == set(source), "Destination state differs from the validated whole-layer boundary")
    for key in state:
        require(state[key].shape == source[key].shape and state[key].dtype == source[key].dtype,
                f"Destination packing/dtype mismatch {mode}/{j}/{key}")
    return [(state[k], source[k]) for k in sorted(state)]

def verify_layer(mode, j):
    state = MODEL.model.layers[j].state_dict()
    prefix = f"model.layers.{j}."
    entries = MANIFEST["layers"][j]["variants"][mode]["all_layer_tensors"]
    require(set(state) == {e["name"][len(prefix):] for e in entries}, "Runtime key set mismatch")
    for entry in entries:
        t = state[entry["name"][len(prefix):]]
        require(t.device == DEVICE and not t.is_meta, "Wrong runtime device")
        require(tensor_digest(t) == entry["normalized_sha256"], f"Runtime value mismatch: {entry['name']}")
    from awq.modules.linear.gemm import WQLinear_GEMM
    for path in LINEARS:
        module = MODEL.model.layers[j].get_submodule(path)
        require(isinstance(module, WQLinear_GEMM if mode == "awq" else torch.nn.Linear), "Dispatch class mismatch")

def ownership_check():
    owners = {}
    for name, t in list(MODEL.named_parameters(remove_duplicate=False)) + list(MODEL.named_buffers(remove_duplicate=False)):
        require(t.device == DEVICE and not t.is_meta, f"Wrong device: {name}")
        if name.startswith("model.layers.") and t.is_floating_point():
            require(t.dtype == torch.float16, f"Unexpected float dtype: {name}")
        scope = int(name.split(".")[2]) if name.startswith("model.layers.") else None
        if t.numel():
            key = (t.untyped_storage().data_ptr(), t.untyped_storage().nbytes())
            owners.setdefault(key, []).append((name, scope))
    for same in owners.values():
        require(len({s for n, s in same}) == 1, "Storage shared across decoder boundaries")
        if same[0][1] is None and len(same) > 1:
            require({n for n, s in same} <= {"model.embed_tokens.weight", "lm_head.weight"}, "Unexpected global alias")
    require(MODEL.get_input_embeddings().weight.data_ptr() == MODEL.get_output_embeddings().weight.data_ptr(), "Expected tied embeddings")
    return {"unique_tensor_backing_bytes": sum(size for ptr, size in owners),
            "all_state_on_gpu_0": True, "awq_layers": sorted(ACTIVE_AWQ)}

def memory_snapshot(label, base_ns, **extra):
    # Instrumented memory diagnostics are separate from latency calibration rows.
    free, total = torch.cuda.mem_get_info(DEVICE)
    return {"label": label, "host_offset_ms": (time.perf_counter_ns()-base_ns)/1e6,
        "allocated_bytes": torch.cuda.memory_allocated(DEVICE), "reserved_bytes": torch.cuda.memory_reserved(DEVICE),
        "peak_allocated_bytes": torch.cuda.max_memory_allocated(DEVICE),
        "peak_reserved_bytes": torch.cuda.max_memory_reserved(DEVICE),
        "whole_device_used_bytes_point_sample": total-free, "whole_device_total_bytes": total,
        "whole_device_peak": "not_measured", **extra}

@torch.inference_mode()
def switch_once(mode, j, prior_reader=None, memory_probe=False):
    global EXECUTOR_HEALTHY
    require(EXECUTOR_HEALTHY, "Executor failed; restart and resume saved cases")
    assert_native()
    source_mode = "awq" if j in ACTIVE_AWQ else "fp16"
    require(mode != source_mode, "Transition must change the layer representation")
    before_config = config_identity(ACTIVE_AWQ)
    source_ledger = (allocation_ledger(f"probe_source_{source_mode}_layer_{j}", MODEL.model.layers[j])
                     if memory_probe else None)
    before_alloc = torch.cuda.memory_allocated(DEVICE)
    torch.cuda.reset_peak_memory_stats(DEVICE)
    events = [torch.cuda.Event(enable_timing=True) for _ in range(3)]
    copy_start, copy_end, committed = events
    # Materialize event handles outside the timed region to exclude first-use setup.
    for event in events:
        event.record(COPY_STREAM)
    COPY_STREAM.synchronize()
    # With pending work on COMPUTE_STREAM, synchronizing COPY_STREAM does not wait on it.
    snapshots = []
    new_layer = old = bindings = None
    t0 = time.perf_counter_ns()
    try:
        if memory_probe:
            snapshots.append(memory_snapshot("before_prior_reader_wait", t0))
        wait0 = time.perf_counter_ns()
        if prior_reader is None:
            COMPUTE_STREAM.synchronize()
        else:
            prior_reader.synchronize()
        wait1 = time.perf_counter_ns()
        if memory_probe:
            snapshots.append(memory_snapshot("prior_readers_finished", t0))
        prepare0 = time.perf_counter_ns()
        new_layer = skeleton(mode, j)
        prepare1 = time.perf_counter_ns()
        with torch.cuda.stream(COPY_STREAM):
            if prior_reader is not None:
                COPY_STREAM.wait_event(prior_reader)
            alloc0 = time.perf_counter_ns()
            new_layer.to_empty(device=DEVICE)
            alloc1 = time.perf_counter_ns()
            bindings = layer_bindings(new_layer, mode, j)
            prepare2 = time.perf_counter_ns()
            actual_copy_bytes = sum(src.numel()*src.element_size() for dst, src in bindings)
            require(actual_copy_bytes == variant_bytes(mode, j), "Actual copy payload mismatch")
            if memory_probe:
                snapshots.append(memory_snapshot("destination_allocated_old_still_live", t0,
                    destination_tensor_bytes=actual_copy_bytes))
            submit0 = time.perf_counter_ns()
            copy_start.record(COPY_STREAM)
            for dst, src in bindings:
                dst.copy_(src, non_blocking=True)
            copy_end.record(COPY_STREAM)
            submit1 = time.perf_counter_ns()
        copy_end.synchronize()
        copy_observed = time.perf_counter_ns()
        destination_ledger = (allocation_ledger(f"probe_destination_{mode}_layer_{j}", new_layer)
                              if memory_probe else None)
        if memory_probe:
            snapshots.append(memory_snapshot("copy_finished_old_and_destination_live", t0))
        commit0 = time.perf_counter_ns()
        COMPUTE_STREAM.wait_event(copy_end)
        old = MODEL.model.layers[j]
        old_ref = weakref.ref(old)
        MODEL.model.layers[j] = new_layer
        if mode == "awq":
            ACTIVE_AWQ.add(j)
        else:
            ACTIVE_AWQ.remove(j)
        commit1 = time.perf_counter_ns()
        if memory_probe:
            snapshots.append(memory_snapshot("assignment_committed_old_reference_retained", t0))
        release0 = time.perf_counter_ns()
        del old
        old = None
        release1 = time.perf_counter_ns()
        require(old_ref() is None, "Old layer still referenced; cannot credit its release")
        committed.record(COMPUTE_STREAM)
        committed.synchronize()
        end = time.perf_counter_ns()
        after_alloc = torch.cuda.memory_allocated(DEVICE)
        if memory_probe:
            snapshots.append(memory_snapshot("old_released_destination_steady", t0))
            require(source_ledger["valid"] and destination_ledger["valid"], "Memory probe allocation ledger is invalid")
            require(source_ledger["model_tensor_storage_bytes"] == variant_bytes(source_mode, j)
                    and destination_ledger["model_tensor_storage_bytes"] == variant_bytes(mode, j),
                    "Memory probe tensor backing differs from its variant manifest")
            expected_alloc = (before_alloc + destination_ledger["model_allocator_block_bytes"]
                              - source_ledger["model_allocator_block_bytes"])
            require(after_alloc == expected_alloc,
                f"Memory probe allocation mismatch: before={before_alloc}, expected_after={expected_alloc}, "
                f"observed_after={after_alloc}; inspect source/destination allocation ledgers")
        row = {"layer": j, "direction": f"{source_mode}_to_{mode}",
            "instrumentation": "memory_probe_not_latency" if memory_probe else "timing",
            "source_configuration": before_config, "destination_configuration": config_identity(ACTIVE_AWQ),
            "host_pinned": True, "actual_copied_bytes": actual_copy_bytes, "copy_tensor_count": len(bindings),
            "old_tensor_bytes": variant_bytes(source_mode, j), "destination_tensor_bytes": variant_bytes(mode, j),
            "net_resident_change_bytes": variant_bytes(mode, j)-variant_bytes(source_mode, j),
            "allocation_layout": CFG["allocation_layout"], "destination_reservation": "full_destination_tensor_backing",
            "prior_reader_wait_host_ms": (wait1-wait0)/1e6,
            "module_structure_preparation_host_ms": (prepare1-prepare0)/1e6,
            "destination_allocation_host_ms": (alloc1-alloc0)/1e6,
            "bindings_preparation_host_ms": (prepare2-alloc1)/1e6,
            "copy_submission_host_ms": (submit1-submit0)/1e6,
            "copy_completion_wait_host_ms": (copy_observed-submit1)/1e6,
            "gpu_copy_interval_ms": copy_start.elapsed_time(copy_end),
            "commit_assignment_host_ms": (commit1-commit0)/1e6,
            "old_release_host_ms": (release1-release0)/1e6,
            "total_synchronized_host_ms": (end-t0)/1e6,
            "transition_after_prior_wait_host_ms": (end-wait1)/1e6,
            "allocated_before_bytes": before_alloc, "allocated_after_bytes": after_alloc,
            "peak_allocated_bytes": torch.cuda.max_memory_allocated(DEVICE),
            "peak_reserved_bytes": torch.cuda.max_memory_reserved(DEVICE),
            "reserved_after_bytes": torch.cuda.memory_reserved(DEVICE),
            "old_object_released": True, "backend_repack": "not_required_for_unfused_GEMM",
            "remapping_cost": "unmeasured", "dependencies": ["prior_reader_done -> copy_start",
                "copy_done -> assignment_commit", "prior_reader_done AND assignment_commit -> old_release",
                "copy_done -> compute_stream_ready"],
            "components_additive": False}
        if prior_reader is not None:
            row["prior_done_to_copy_start_gpu_ms"] = prior_reader.elapsed_time(copy_start)
            row["prior_done_to_committed_ready_gpu_ms"] = prior_reader.elapsed_time(committed)
        if memory_probe:
            row.update({"source_allocator_block_bytes": source_ledger["model_allocator_block_bytes"],
                "destination_allocator_block_bytes": destination_ledger["model_allocator_block_bytes"],
                "expected_allocated_after_bytes": expected_alloc,
                "source_allocation_ledger": source_ledger["report_file"],
                "destination_allocation_ledger": destination_ledger["report_file"]})
        return row, snapshots
    except BaseException:
        EXECUTOR_HEALTHY = False
        try:
            COPY_STREAM.synchronize()
            COMPUTE_STREAM.synchronize()
        finally:
            new_layer = old = bindings = None
        raise

def set_layer(mode, j):
    if ("awq" if j in ACTIVE_AWQ else "fp16") != mode:
        switch_once(mode, j)

def reset_fp16():
    for j in sorted(ACTIVE_AWQ, reverse=True):
        set_layer("fp16", j)
    torch.cuda.synchronize(DEVICE)

@contextmanager
def serial_awq_for_correctness_only():
    original = GEMM.awq_gemm_triton
    def serial(x, qweight, scales, qzeros, split_k_iters=8, **kwargs):
        return original(x, qweight, scales, qzeros, split_k_iters=1, **kwargs)
    torch.cuda.synchronize(DEVICE)
    GEMM.awq_gemm_triton = serial
    try:
        yield
    finally:
        try:
            torch.cuda.synchronize(DEVICE)
        finally:
            GEMM.awq_gemm_triton = original


## 6. Freeze disjoint calibration, fitting and validation inputs

WikiText's first train-token half supplies LIS windows; the second half supplies operation-fitting windows.
The separate validation split supplies held-out timing inputs. Windows do not overlap within these pools.
Token IDs, source spans, dataset commit/file hashes, masks and scoring positions are saved before scoring.
The test split and final scheduler evaluation data remain untouched.


In [ ]:
DATA = None
ORDER = None

def frozen_window_plan(token_count, length, count, seed, lower=0, upper=None):
    upper = token_count if upper is None else upper
    require(0<=lower<upper<=token_count and length>0, "Invalid token partition")
    starts = list(range(lower,upper-length+1,length))
    require(len(starts)>=count, "Dataset partition is too short")
    return random.Random(seed).sample(starts,count)

def tensor_tokens_digest(ids):
    return hashlib.sha256(np.asarray(ids,dtype="<i8").tobytes()).hexdigest()

def data_stage():
    global DATA
    path = OUT/"frozen_inputs.json"
    if path.exists():
        DATA = json.loads(path.read_text())
        require(DATA["configuration_sha256"]==CONFIG_HASH, "Frozen input configuration mismatch")
        raw = {k:v for k,v in DATA.items() if k!="content_sha256"}
        require(DATA["content_sha256"]==canonical_hash(raw), "Frozen input file was edited")
        return DATA["provenance"]
    import pyarrow.parquet as pq
    spec = CFG["dataset"]
    streams, records = {},{}
    for split in ["train","validation"]:
        filename = f"{spec['subset']}/{split}-00000-of-00001.parquet"
        local = Path(hf_hub_download(repo_id=spec["repo"],repo_type="dataset",revision=spec["revision"],filename=filename))
        texts = pq.read_table(local,columns=["text"])["text"].to_pylist()
        # One declared canonical plain-text stream; no chat formatting or padding.
        joined = "\n\n".join(str(x) for x in texts if x is not None)
        require(hasattr(TOKENIZER,"backend_tokenizer"), "Expected the pinned fast tokenizer")
        TOKENIZER.backend_tokenizer.no_truncation()
        TOKENIZER.backend_tokenizer.no_padding()
        streams[split] = TOKENIZER.backend_tokenizer.encode(joined,add_special_tokens=False).ids
        records[split] = {"file":filename,"parquet_sha256":hashlib.sha256(local.read_bytes()).hexdigest(),
            "joined_utf8_sha256":hashlib.sha256(joined.encode()).hexdigest(),"tokens":len(streams[split])}
    train = streams["train"]
    middle = len(train)//2
    maximum_length = max(max(CFG["lengths"]),max(n for b,n in CFG["anchor_shapes"]))
    maximum_batch = max(max(CFG["batches"]),max(b for b,n in CFG["anchor_shapes"]))
    cal_starts = frozen_window_plan(len(train),CFG["calibration_length"],CFG["calibration_examples"],CFG["seed"],upper=middle)
    fit_starts = frozen_window_plan(len(train),maximum_length,maximum_batch,CFG["seed"]+1,lower=middle)
    val_starts = frozen_window_plan(len(streams["validation"]),maximum_length,maximum_batch,CFG["seed"]+2)
    def windows(stream, starts, length, split):
        return [{"split":split,"start":start,"end_exclusive":start+length,
            "token_ids":stream[start:start+length],"token_ids_sha256":tensor_tokens_digest(stream[start:start+length])}
            for start in starts]
    calibration = windows(train,cal_starts,CFG["calibration_length"],"train_first_half")
    fit = windows(train,fit_starts,maximum_length,"train_second_half")
    validation = windows(streams["validation"],val_starts,maximum_length,"validation")
    require(not({w["token_ids_sha256"] for w in calibration}&{w["token_ids_sha256"] for w in fit+validation}), "Duplicate calibration/profiling window")
    require(not({w["token_ids_sha256"] for w in fit}&{w["token_ids_sha256"] for w in validation}), "Duplicate fitting/validation window")
    provenance = {"dataset":spec,"source_files":records,"train_partition_boundary_token":middle,
        "calibration_examples":len(calibration),"calibration_length":CFG["calibration_length"],
        "positions":sampled_positions(CFG["calibration_length"],CFG["score_positions_per_example"]),
        "scoring_dispatch":"dequantize_matmul" if CFG["calibration_length"]>=1024 else "GEMM_splitK1",
        "splitK_override_scope":"GEMM only; at the default 2048-token scoring length the large-token dequantize/matmul path is unchanged",
        "mask":"all ones; fixed-length, non-overlapping windows; no padding",
        "plain_text_join":"all text rows joined with two LF characters; no chat template or special tokens",
        "split_rule":"LIS: train first half; cost fitting: train second half; validation: separate validation split",
        "test_split":"untouched; final scheduler trace/evaluation also must remain disjoint",
        "license":"WikiText: CC BY-SA 3.0 / GFDL; includes Wikipedia-derived text. Preserve attribution."}
    DATA = {"configuration_sha256":CONFIG_HASH,"provenance":provenance,
            "calibration":calibration,"fit":fit,"validation":validation}
    DATA["content_sha256"] = canonical_hash(DATA)
    atomic_json(path,DATA)
    (OUT/"DATA_ATTRIBUTION.md").write_text(
        "WikiText by Stephen Merity, Caiming Xiong, James Bradbury and Richard Socher (2016).\n"
        "Dataset: https://huggingface.co/datasets/Salesforce/wikitext\n"
        "Wikipedia-derived content; CC BY-SA 3.0 / GFDL, as declared by the dataset card.\n"
        "Only the pinned train/validation files and recorded token windows were used.\n")
    return provenance

run_stage("data",data_stage)


## 7. Check the selected-logit API and original FP16 reference

Only the chosen positions' final vocabulary logits are produced for ranking, which avoids saving full
2,048-position vocabulary outputs. Calibration remains a full-context forward pass; positions are sampled
only for comparison. Ranking selects serial split-K 1 wherever the GEMM branch executes; native performance
measurements restore split-K 8. With default 2,048-token calibration, AWQ uses its unchanged large-token
dequantize/matmul branch, so the split-K override is dormant on that branch.


In [ ]:
def cpu_tree(value):
    if isinstance(value,torch.Tensor):
        return value.detach().cpu().clone()
    if isinstance(value,dict):
        return {k:cpu_tree(v) for k,v in value.items()}
    if isinstance(value,tuple):
        return tuple(cpu_tree(v) for v in value)
    if isinstance(value,list):
        return [cpu_tree(v) for v in value]
    require(value is None or isinstance(value,(str,int,float,bool)), "Unsupported decoder argument in replay")
    return value

def gpu_tree(value):
    if isinstance(value,torch.Tensor):
        return value.to(DEVICE)
    if isinstance(value,dict):
        return {k:gpu_tree(v) for k,v in value.items()}
    if isinstance(value,tuple):
        return tuple(gpu_tree(v) for v in value)
    if isinstance(value,list):
        return [gpu_tree(v) for v in value]
    return value

def set_configuration(layers, verify=False):
    target = set(layers)
    require(target<=set(range(28)), "Invalid layer set")
    for j in sorted(ACTIVE_AWQ-target,reverse=True):
        set_layer("fp16",j)
        if verify:
            verify_layer("fp16",j)
    for j in sorted(target-ACTIVE_AWQ):
        set_layer("awq",j)
        if verify:
            verify_layer("awq",j)
    require(ACTIVE_AWQ==target, "Committed configuration mismatch")

def output_hidden(result):
    result = result[0] if isinstance(result,(tuple,list)) else result
    require(isinstance(result,torch.Tensor), "Unexpected decoder output structure")
    return result

def cache_pairs(cache):
    require(hasattr(cache,"to_legacy_cache"), "Expected DynamicCache")
    result = cache.to_legacy_cache()
    require(len(result)==28, "Unexpected KV layer count")
    return result

def check_cache(cache, length, batch=1):
    require(cache.get_seq_length()==length, "Unexpected cache sequence length")
    expected = (batch,MODEL_CONFIG.num_key_value_heads,length,
                MODEL_CONFIG.hidden_size//MODEL_CONFIG.num_attention_heads)
    logical,storage_sizes,refs = 0,{},[]
    for j,pair in enumerate(cache_pairs(cache)):
        for kind,t in zip(["key","value"],pair):
            require(tuple(t.shape)==expected and t.device==DEVICE and t.dtype==torch.float16, "KV shape/device/dtype mismatch")
            logical += t.numel()*t.element_size()
            storage = t.untyped_storage()
            storage_sizes[storage.data_ptr()] = storage.nbytes()
            refs.append((f"cache_layer_{j}_{kind}",weakref.ref(t)))
    expected_bytes = batch*length*28672
    require(logical==expected_bytes, "KV logical byte count mismatch")
    return {"logical_KV_bytes":logical,"unique_KV_storage_bytes":sum(storage_sizes.values())},refs

@torch.inference_mode()
def selected_logits(example, serial=True):
    ids = torch.tensor([example["token_ids"]],dtype=torch.long,device=DEVICE)
    positions = torch.tensor(DATA["provenance"]["positions"],dtype=torch.long,device=DEVICE)
    def forward():
        result = MODEL(input_ids=ids,attention_mask=torch.ones_like(ids),use_cache=False,
            return_dict=True,logits_to_keep=positions)
        require(result.logits.shape==(1,len(positions),MODEL_CONFIG.vocab_size), "Selected-logit API/shape mismatch")
        require(bool(torch.isfinite(result.logits).all()), "Nonfinite calibration logits")
        return result.logits.detach().cpu()
    if serial:
        with serial_awq_for_correctness_only():
            result = forward()
    else:
        assert_native()
        result = forward()
    return result

ORIGINAL_TRACE = None
TIMING_EVENTS = None

def bootstrap():
    global ORIGINAL_TRACE,TIMING_EVENTS
    reset_fp16()
    for j in range(28):
        verify_layer("fp16",j)
    require("logits_to_keep" in inspect.signature(MODEL.forward).parameters, "Unexpected Qwen2 API")
    ORIGINAL_TRACE = selected_logits(DATA["calibration"][0])
    repeat = selected_logits(DATA["calibration"][0])
    require(torch.equal(ORIGINAL_TRACE,repeat), "FP16 calibration repeat was not bitwise repeatable")
    assert_native()
    TIMING_EVENTS = [torch.cuda.Event(enable_timing=True),torch.cuda.Event(enable_timing=True)]
    for event in TIMING_EVENTS:
        event.record(COMPUTE_STREAM)
    COMPUTE_STREAM.synchronize()
    return {"FP16_selected_logits_repeatable":True,"one_active_GPU_model":True,
            "score_path_split_k":1,"performance_path_split_k":8,"ownership":ownership_check()}

run_stage("bootstrap",bootstrap)


## 8. Compute local scores, then greedily freeze the full order

LTS compares original layer input/output. LRS replays the **full original context and rotary/attention arguments**
through each layer variant before sampling outputs. FP16 replay is checked against the original capture.

Each greedy round recomputes MDS against the current quantized prefix using final model logits. Choose the
highest `0.25 LTS + 0.25 LRS + 0.50 MDS`, with ascending layer index for exact ties. A one-time sorted list
does not implement this method. Every candidate/example and each completed round is saved.

This cell is the longest ranking step. It logs progress after each round. Large activation caches are
temporary reproduction aids and are excluded from the final ZIP; raw scores and frozen inputs are retained.


In [ ]:
def activation_path(sample, layer):
    return CACHE/"activations"/f"sample_{sample:02d}_layer_{layer:02d}.pt"

@torch.inference_mode()
def capture_fp16_activations():
    reset_fp16()
    require(shutil.disk_usage(OUT).free > 8*1024**3, "Keep at least 8 GiB free for checkpoints and calibration activation scratch")
    records = []
    for sample,example in enumerate(DATA["calibration"]):
        index_path = CACHE/"activations"/f"sample_{sample:02d}_index.json"
        if index_path.exists():
            index = json.loads(index_path.read_text())
            require(index["token_ids_sha256"]==example["token_ids_sha256"] and index["configuration_sha256"]==CONFIG_HASH,
                    "Activation cache identity mismatch")
            if all(activation_path(sample,j).exists() and
                   hashlib.sha256(activation_path(sample,j).read_bytes()).hexdigest()==index["files"][str(j)] for j in range(28)):
                records.append(index)
                continue
        positions = torch.tensor(DATA["provenance"]["positions"],dtype=torch.long,device=DEVICE)
        seen = set()
        def make_hook(j):
            def hook(module,args,kwargs,output):
                require(len(args)<=1, "Decoder replay needs a review for additional positional arguments")
                hidden = args[0] if args else kwargs.get("hidden_states")
                require(isinstance(hidden,torch.Tensor), "Missing original decoder hidden inputs")
                replay = dict(kwargs)
                replay.pop("hidden_states",None)
                require(replay.get("past_key_value") is None, "Static LRS must not reuse a request cache")
                require(not replay.get("use_cache",False), "Static replay unexpectedly enabled cache")
                original = output_hidden(output)
                selected_input = hidden[:,positions,:].detach().cpu()
                selected_output = original[:,positions,:].detach().cpu()
                payload = {"hidden_states":hidden.detach().cpu().clone(), "kwargs":cpu_tree(replay),
                    "selected_output":selected_output,"lts_tokens":cosine_tokens(selected_input.numpy(),selected_output.numpy()),
                    "configuration_sha256":CONFIG_HASH,"sample":sample,"layer":j,
                    "token_ids_sha256":example["token_ids_sha256"]}
                atomic_torch(activation_path(sample,j),payload)
                seen.add(j)
            return hook
        handles = []
        try:
            handles = [layer.register_forward_hook(make_hook(j),with_kwargs=True) for j,layer in enumerate(MODEL.model.layers)]
            selected_logits(example)
            require(seen==set(range(28)), "Not all FP16 decoder activations were captured")
        finally:
            for handle in handles:
                handle.remove()
        index = {"configuration_sha256":CONFIG_HASH,"token_ids_sha256":example["token_ids_sha256"],
            "files":{str(j):hashlib.sha256(activation_path(sample,j).read_bytes()).hexdigest() for j in range(28)}}
        atomic_json(index_path,index)
        records.append(index)
        print(f"FP16 calibration activations {sample+1}/{len(DATA['calibration'])} saved",flush=True)
    atomic_json(OUT/"activation_cache_manifest.json",{"samples":records,
        "archive_policy":"Large activation tensors are scratch, excluded from ZIP; reconstruct from frozen inputs if needed"})

def load_activation(sample, j):
    payload = torch.load(activation_path(sample,j),map_location="cpu",weights_only=True)
    require(payload["configuration_sha256"]==CONFIG_HASH and payload["sample"]==sample and payload["layer"]==j,
            "Incorrect activation replay identity")
    return payload

@torch.inference_mode()
def replay_selected(payload, j, serial):
    hidden = payload["hidden_states"].to(DEVICE)
    kwargs = gpu_tree(payload["kwargs"])
    positions = torch.tensor(DATA["provenance"]["positions"],dtype=torch.long,device=DEVICE)
    def forward():
        result = output_hidden(MODEL.model.layers[j](hidden,**kwargs))
        result = result[:,positions,:]
        require(bool(torch.isfinite(result).all()), "Nonfinite layer replay")
        return result.detach().cpu()
    if serial:
        with serial_awq_for_correctness_only():
            result = forward()
    else:
        result = forward()
    return result

def static_layer_case(case):
    j = case["layer"]
    reset_fp16()
    first = load_activation(0,j)
    replayed = replay_selected(first,j,serial=False)
    require(torch.allclose(replayed,first["selected_output"],atol=.02,rtol=.002),
            "Direct FP16 replay differs from captured full-forward output; stop rather than score mismatched contexts")
    case["fp16_replay_max_absolute_difference"] = float((replayed.float()-first["selected_output"].float()).abs().max())
    try:
        set_layer("awq",j)
        verify_layer("awq",j)
        for sample,example in enumerate(DATA["calibration"]):
            payload = first if sample==0 else load_activation(sample,j)
            actual = replay_selected(payload,j,serial=True)
            lrs = cosine_tokens(payload["selected_output"].numpy(),actual.numpy())
            case["rows"].append({"sample":sample,"token_ids_sha256":example["token_ids_sha256"],
                "positions":DATA["provenance"]["positions"],"lts_tokens":payload["lts_tokens"],"lrs_tokens":lrs})
            checkpoint(case)
    finally:
        if EXECUTOR_HEALTHY:
            set_layer("fp16",j)
    verify_layer("fp16",j)
    case["lts"] = statistics.fmean(x for r in case["rows"] for x in r["lts_tokens"])
    case["lrs"] = statistics.fmean(x for r in case["rows"] for x in r["lrs_tokens"])
    case["scored_token_count"] = sum(len(r["lts_tokens"]) for r in case["rows"])

def static_scores_stage():
    missing = [j for j in range(28) if passed_case(f"static_layer_{j:02d}") is None]
    if missing:
        capture_fp16_activations()
    for j in missing:
        case = new_case(f"static_layer_{j:02d}","static",layer=j)
        finish_case(case,static_layer_case)
        print(f"LTS/LRS layer {j}: {case['lts']:.8f}, {case['lrs']:.8f}",flush=True)
    reset_fp16()
    return {"layers":28,"calibration_examples":len(DATA["calibration"]),
        "selected_positions":DATA["provenance"]["positions"],"full_context_replayed_for_LRS":True,
        "scoring_kernel_split_k":1,"all_hooks_removed":True}

run_stage("static_scores",static_scores_stage)

def validate_order_record(record):
    require(record["configuration_sha256"]==CONFIG_HASH, "Order belongs to another configuration")
    order = record["order"]
    require(len(order)==len(set(order)) and set(order)<=set(range(28)), "Invalid partial layer permutation")
    require(record["content_sha256"]==canonical_hash({k:v for k,v in record.items() if k!="content_sha256"}), "Frozen order file hash mismatch")
    return order

def save_order(order, rounds):
    record = {"schema_version":1,"status":"FROZEN" if len(order)==28 else "PARTIAL",
        "configuration_sha256":CONFIG_HASH,"frozen_inputs_sha256":DATA["content_sha256"],
        "order":list(order),"restoration_order":list(reversed(order)),"rounds":rounds,
        "method":"greedy argmax; LTS/LRS fixed on original FP16 inputs; incremental MDS at the current prefix",
        "weights":CFG["score_weights"],"tie_rule":"exact score ties use ascending decoder layer index",
        "score_positions":DATA["provenance"]["positions"],"scoring_kernel":CFG["scoring_kernel"],
        "performance_kernel":CFG["performance_kernel"],
        "prefix_configurations":[{"q":q,**config_identity(order[:q])} for q in range(len(order)+1)]}
    record["content_sha256"] = canonical_hash(record)
    atomic_json(OUT/"layer_order.json",record)
    return record

def greedy_candidate_case(case, references):
    j = case["layer"]
    try:
        set_layer("awq",j)
        verify_layer("awq",j)
        for sample,example in enumerate(DATA["calibration"]):
            actual = selected_logits(example)
            values = cosine_tokens(references[sample].numpy(),actual.numpy())
            case["rows"].append({"sample":sample,"token_ids_sha256":example["token_ids_sha256"],
                "positions":DATA["provenance"]["positions"],"mds_tokens":values})
            checkpoint(case)
    finally:
        if EXECUTOR_HEALTHY:
            set_layer("fp16",j)
    verify_layer("fp16",j)
    case["mds"] = statistics.fmean(x for r in case["rows"] for x in r["mds_tokens"])
    case["lis"] = lis_score(case["lts"],case["lrs"],case["mds"])

def greedy_stage():
    global ORDER
    order,rounds = [],[]
    if (OUT/"layer_order.json").exists():
        record = json.loads((OUT/"layer_order.json").read_text())
        order = validate_order_record(record)
        rounds = record["rounds"]
    # Revalidate any completed prefix against its saved candidate scores before reuse.
    for q,j in enumerate(order):
        candidates = [passed_case(f"greedy_q{q:02d}_layer_{k:02d}") for k in range(28) if k not in order[:q]]
        require(all(c is not None for c in candidates) and greedy_winner(candidates)["layer"]==j,
                "Saved greedy prefix is inconsistent with its complete candidate records")
    while len(order)<28:
        q = len(order)
        set_configuration(order,verify=True)
        references = [selected_logits(e) for e in DATA["calibration"]]
        repeat = selected_logits(DATA["calibration"][0])
        require(torch.equal(repeat,references[0]), "Current-prefix selected logits are not repeatable under serial reduction")
        reference_hashes = [tensor_digest(x) for x in references]
        candidates = []
        for j in range(28):
            if j in order:
                continue
            cid = f"greedy_q{q:02d}_layer_{j:02d}"
            case = passed_case(cid)
            if case is None:
                static = passed_case(f"static_layer_{j:02d}")
                require(static is not None, "Missing fixed local sensitivity scores")
                case = new_case(cid,"greedy",q=q,layer=j,prefix=list(order),
                    prefix_configuration=config_identity(order),reference_logits_sha256=reference_hashes,
                    lts=static["lts"],lrs=static["lrs"])
                finish_case(case,lambda c:greedy_candidate_case(c,references))
            else:
                require(case["prefix"]==order and case["reference_logits_sha256"]==reference_hashes,
                        "Candidate cache does not match the prefix/reference logits")
            require(ACTIVE_AWQ==set(order), "Candidate evaluation did not restore its previous prefix")
            candidates.append(case)
        chosen = greedy_winner(candidates)
        ranked = sorted(candidates,key=lambda r:(-r["lis"],r["layer"]))
        order.append(chosen["layer"])
        set_layer("awq",chosen["layer"])
        verify_layer("awq",chosen["layer"])
        rounds.append({"q_before":q,"selected_layer":chosen["layer"],"candidate_count":len(candidates),
            "winning_lis":chosen["lis"],"winning_margin":chosen["lis"]-ranked[1]["lis"] if len(ranked)>1 else None,
            "reference_logits_sha256":reference_hashes,"repeatability_check":"example 0 bitwise repeat under serial split-K 1"})
        save_order(order,rounds)
        print(f"Greedy round {q+1}/28: chose layer {chosen['layer']}; LIS={chosen['lis']:.9f}",flush=True)
    ORDER = list(order)
    require(set(ORDER)==set(range(28)), "The frozen order is incomplete")
    reset_fp16()
    return {"order":ORDER,"restore_order":list(reversed(ORDER)),"candidate_configurations":28*29//2,
            "order_file":"layer_order.json","path_sha256":json.loads((OUT/"layer_order.json").read_text())["content_sha256"]}

run_stage("greedy_order",greedy_stage)


## 9. Validate the frozen path and collect all configuration footprints

A live request survives precision changes and token appends along the frozen order and reverse restoration.
Cached prefixes, token counts, original weights and fresh FP16 logits are checked. Separately, all q=0…28
footprints are measured after native warmup with no live request. Tensor backing, allocator blocks,
allocated/reserved memory and native backend overhead are kept separate. This does not measure elastic KV pages.


In [ ]:
def prefix_copy(cache):
    return [(k.detach().cpu().clone(),v.detach().cpu().clone()) for k,v in cache_pairs(cache)]

def prefix_unchanged(cache, snapshot):
    for (k,v),(oldk,oldv) in zip(cache_pairs(cache),snapshot):
        n = oldk.shape[-2]
        require(torch.equal(k[...,:n,:].cpu(),oldk) and torch.equal(v[...,:n,:].cpu(),oldv), "Previously cached state changed")

@torch.inference_mode()
def append_native(cache):
    assert_native()
    n = cache.get_seq_length()
    out = MODEL(input_ids=torch.tensor([[42]],dtype=torch.long,device=DEVICE),
        attention_mask=torch.ones((1,n+1),dtype=torch.long,device=DEVICE),
        position_ids=torch.tensor([[n]],device=DEVICE),cache_position=torch.tensor([n],device=DEVICE),
        past_key_values=cache,use_cache=True,return_dict=True,logits_to_keep=1)
    require(out.past_key_values is cache and bool(torch.isfinite(out.logits).all()), "Native resume/cache mismatch")
    return out

@torch.inference_mode()
def native_path_cycle():
    reset_fp16()
    ids = torch.tensor([DATA["fit"][0]["token_ids"][:16]],dtype=torch.long,device=DEVICE)
    out = MODEL(input_ids=ids,attention_mask=torch.ones_like(ids),use_cache=True,return_dict=True,logits_to_keep=1)
    cache = out.past_key_values
    n = len(ids[0])
    snapshot = prefix_copy(cache)
    refs = [("initial_ids",weakref.ref(ids)),("initial_logits",weakref.ref(out.logits))]
    events = []
    for dropping,qs in [(True,range(1,29)),(False,range(27,-1,-1))]:
        for q in qs:
            j = ORDER[q-1] if dropping else ORDER[q]
            set_layer("awq" if dropping else "fp16",j)
            if q in [0,1,7,14,21,28]:
                prefix_unchanged(cache,snapshot)
                out = append_native(cache)
                n += 1
                check_cache(cache,n)
                prefix_unchanged(cache,snapshot)
                snapshot = prefix_copy(cache)
                refs.append((f"native_q{q}_logits",weakref.ref(out.logits)))
                refs.extend((f"native_q{q}_kv_{j}_{kind}",weakref.ref(t)) for j,pair in enumerate(cache_pairs(cache))
                            for kind,t in zip(["k","v"],pair))
                events.append({"direction":"drop" if dropping else "restore","q":q,
                    "cached_positions":n,**config_identity(ACTIVE_AWQ)})
    require(not ACTIVE_AWQ, "Native path cycle did not restore q=0")
    return events,refs

def path_validation_stage():
    events,refs = native_path_cycle()
    gc.collect()
    torch.cuda.synchronize(DEVICE)
    live = [label for label,ref in refs if ref() is not None]
    require(not live, f"Native path validation retained request tensors: {live}")
    for j in range(28):
        verify_layer("fp16",j)
    restored = selected_logits(DATA["calibration"][0])
    require(torch.equal(restored,ORIGINAL_TRACE), "Fresh FP16 logits changed after the frozen-path cycle")
    result = {"events":events,"native_split_k":8,"retained_prefixes_unchanged":True,
        "all_original_weights_exact":True,"fresh_FP16_logits_exact":True,"tracked_request_tensors_released":True}
    atomic_json(OUT/"path_validation.json",result)
    return {k:v for k,v in result.items() if k!="events"}

run_stage("path_validation",path_validation_stage)

@torch.inference_mode()
def native_weight_warmup():
    assert_native()
    ids = torch.tensor([DATA["fit"][0]["token_ids"][:16]],dtype=torch.long,device=DEVICE)
    out = MODEL(input_ids=ids,attention_mask=torch.ones_like(ids),use_cache=False,return_dict=True,logits_to_keep=1)
    require(bool(torch.isfinite(out.logits).all()), "Nonfinite footprint warmup")

def footprints_stage():
    reset_fp16()
    original_tensor_bytes = ownership_check()["unique_tensor_backing_bytes"]
    rows = []
    for q in range(29):
        set_configuration(ORDER[:q],verify=True)
        native_weight_warmup()
        ledger = allocation_ledger(f"path_q{q:02d}_native_no_KV")
        require(ledger["valid"], "Invalid footprint allocation ledger")
        expected = original_tensor_bytes + sum(variant_bytes("awq",j)-variant_bytes("fp16",j) for j in ORDER[:q])
        require(ledger["model_tensor_storage_bytes"]==expected, "Configuration tensor backing does not match manifests")
        row = {"q":q,**config_identity(ACTIVE_AWQ),"layer_order_sha256":canonical_hash(ORDER),
            "model_tensor_backing_bytes":ledger["model_tensor_storage_bytes"],
            "model_allocator_block_bytes":ledger["model_allocator_block_bytes"],
            "allocated_baseline_bytes":ledger["allocated_bytes"],
            "nonmodel_allocator_block_bytes":ledger["nonmodel_allocator_block_bytes"],
            "reserved_bytes":torch.cuda.memory_reserved(DEVICE),"allocation_ledger":ledger["report_file"],
            "raw_KV_bytes_per_token":28672,"KV_page_alignment":"not_measured",
            "layout":CFG["allocation_layout"],"active_requests":0}
        rows.append(row)
        atomic_json(OUT/"configuration_footprints.json",rows)
    reset_fp16()
    for j in range(28):
        verify_layer("fp16",j)
    return {"configurations":29,"raw_vs_allocator_recorded_separately":True,"page_detachment":"not_measured"}

run_stage("footprints",footprints_stage)


## 10. Profile native intermediate prefill/decode operations

The default fitting configurations are q=1,7,14,21. Timing validation uses q=10,18 and disjoint input data.
All six use batches 1,2,4,8 and lengths 256,512,1,024,2,048, with separate prefill and one-token cached decode.
Targeted q=0/28 anchor checks avoid repeating the old full endpoint grids. Total: **200 cases**.

Each decode trial starts at exactly its declared cached length. Input preparation and decode's prefill setup
are outside the timed operation. GPU event and synchronized host times are recorded independently.
Memory trials add lifecycle snapshots and are excluded from latency summaries. Request state is checked
after each helper returns; allocator ledgers explain cleanup differences without relaxing leak checks.

Do not interrupt a case unless necessary. If a GPU stage fails, export partial results, restart the kernel,
then resume the existing writable run folder with unchanged settings/source.


In [ ]:
def profile_specs():
    result = []
    for role,qs in [("fit",CFG["fit_q"]),("validation",CFG["validation_q"])]:
        for q in qs:
            for batch in CFG["batches"]:
                for length in CFG["lengths"]:
                    for phase in ["prefill","decode"]:
                        result.append({"role":role,"q":q,"batch":batch,"length":length,"phase":phase})
    for q in [0,28]:
        for batch,length in CFG["anchor_shapes"]:
            for phase in ["prefill","decode"]:
                result.append({"role":"anchor","q":q,"batch":batch,"length":length,"phase":phase})
    return result

def profile_id(spec):
    return f"profile_{spec['role']}_q{spec['q']:02d}_{spec['phase']}_b{spec['batch']}_n{spec['length']}"

@torch.inference_mode()
def operation_trial(spec, instrumentation):
    # This function owns all CUDA request state. Only CPU metrics/weakrefs escape.
    assert_native()
    batch,length,phase = spec["batch"],spec["length"],spec["phase"]
    bank = DATA["validation" if spec["role"]=="validation" else "fit"]
    cpu_ids = [bank[i]["token_ids"][:length] for i in range(batch)]
    baseline = torch.cuda.memory_allocated(DEVICE)
    points = []
    base_ns = time.perf_counter_ns()
    if instrumentation=="memory":
        torch.cuda.reset_peak_memory_stats(DEVICE)
        points.append(memory_snapshot("clean_configuration",base_ns))
    ids = torch.tensor(cpu_ids,dtype=torch.long,device=DEVICE)
    args = {"input_ids":ids,"attention_mask":torch.ones_like(ids),"use_cache":True,"return_dict":True,"logits_to_keep":1}
    refs = [("input_ids",weakref.ref(ids)),("input_mask",weakref.ref(args["attention_mask"]))]
    if phase=="decode":
        prefill = MODEL(**args)
        cache = prefill.past_key_values
        _,prior_refs = check_cache(cache,length,batch)
        refs.extend(prior_refs)
        del prefill
        args = {"input_ids":torch.full((batch,1),42,dtype=torch.long,device=DEVICE),
            "attention_mask":torch.ones((batch,length+1),dtype=torch.long,device=DEVICE),
            "position_ids":torch.full((batch,1),length,dtype=torch.long,device=DEVICE),
            "cache_position":torch.tensor([length],dtype=torch.long,device=DEVICE),
            "past_key_values":cache,"use_cache":True,"return_dict":True,"logits_to_keep":1}
        refs.extend((name,weakref.ref(value)) for name,value in args.items() if isinstance(value,torch.Tensor))
    torch.cuda.synchronize(DEVICE)
    allocated_before_operation = torch.cuda.memory_allocated(DEVICE)
    if instrumentation=="memory":
        points.append(memory_snapshot("inputs_and_optional_decode_cache_ready",base_ns))
    # Reset here so decode prefill-setup peaks cannot be mislabeled as decode workspace.
    torch.cuda.reset_peak_memory_stats(DEVICE)
    start,end = TIMING_EVENTS
    start.record(COMPUTE_STREAM)
    wall0 = time.perf_counter_ns()
    out = MODEL(**args)
    end.record(COMPUTE_STREAM)
    end.synchronize()
    wall1 = time.perf_counter_ns()
    operation_peak = torch.cuda.max_memory_allocated(DEVICE)
    operation_peak_reserved = torch.cuda.max_memory_reserved(DEVICE)
    allocated_after_operation = torch.cuda.memory_allocated(DEVICE)
    if instrumentation=="memory":
        points.append(memory_snapshot("operation_complete_live_output_and_KV",base_ns))
    kv,kv_refs = check_cache(out.past_key_values,length+(phase=="decode"),batch)
    refs.extend(kv_refs)
    refs.append(("output_logits",weakref.ref(out.logits)))
    require(bool(torch.isfinite(out.logits).all()), "Nonfinite native profile logits")
    if phase=="decode":
        require(out.past_key_values is cache, "Decode replaced its cache object")
    row = {**spec,**config_identity(ACTIVE_AWQ),"layer_order_sha256":canonical_hash(ORDER),
        "instrumentation":instrumentation,"gpu_operation_ms":start.elapsed_time(end),
        "synchronized_host_operation_ms":(wall1-wall0)/1e6,
        "allocated_clean_baseline_bytes":baseline,"allocated_before_operation_bytes":allocated_before_operation,
        "allocated_after_operation_bytes":allocated_after_operation,
        "operation_peak_allocated_bytes":operation_peak,"operation_peak_reserved_bytes":operation_peak_reserved,
        "operation_peak_includes":"model, live request tensors, cache/output and execution temporaries; not isolated workspace",
        "data_bank":"validation" if spec["role"]=="validation" else "fit",
        "input_ids_sha256":tensor_tokens_digest(cpu_ids),"cached_positions_after":length+(phase=="decode"),
        "batch_advances_per_operation":batch if phase=="decode" else 0,
        "AWQ_dispatch":"none" if not ACTIVE_AWQ else ("dequantize_matmul" if batch*(length if phase=="prefill" else 1)>=1024 else "native_GEMM_splitK8"),
        "separate_phase":True,**kv,"memory_points":points}
    row["_memory_origin_ns"] = base_ns
    return row,refs

def checked_trial(spec, instrumentation, baseline_ledger=None):
    row,refs = operation_trial(spec,instrumentation)
    memory_origin = row.pop("_memory_origin_ns")
    gc.collect()
    torch.cuda.synchronize(DEVICE)
    live = [label for label,ref in refs if ref() is not None]
    require(not live, f"Request state remains live after operation returned: {live}")
    row["allocated_after_cleanup_bytes"] = torch.cuda.memory_allocated(DEVICE)
    row["all_tracked_request_tensors_released"] = True
    if instrumentation=="memory":
        row["memory_points"].append(memory_snapshot("request_released",memory_origin))
    if baseline_ledger is not None and row["allocated_after_cleanup_bytes"]!=baseline_ledger["allocated_bytes"]:
        after = allocation_ledger(f"profile_cleanup_q{spec['q']}_{spec['phase']}_b{spec['batch']}_n{spec['length']}")
        cleanup = compare_cleanup_ledgers(baseline_ledger,after)
        row["cleanup_accounting"] = cleanup
        require(cleanup["passed"], f"Profile cleanup failed: {cleanup}")
    return row

def profile_case(case):
    spec = case["spec"]
    set_configuration(ORDER[:spec["q"]],verify=True)
    for repeat in range(CFG["warmups"]):
        row = checked_trial(spec,"timing")
        row.update({"warmup":True,"repeat":repeat})
        case["rows"].append(row)
        checkpoint(case)
    # Any first-use setup belongs to warmups. Retained request tensors have already been checked.
    baseline = allocation_ledger(profile_id(spec)+"_after_warmup")
    require(baseline["valid"], "Invalid post-warmup allocation ledger")
    case["baseline_ledger"] = baseline["report_file"]
    for repeat in range(CFG["trials"]):
        row = checked_trial(spec,"timing",baseline)
        row.update({"warmup":False,"repeat":repeat})
        case["rows"].append(row)
        checkpoint(case)
    for repeat in range(CFG["memory_trials"]):
        row = checked_trial(spec,"memory",baseline)
        row.update({"warmup":False,"repeat":repeat})
        case["rows"].append(row)
        checkpoint(case)
    final = allocation_ledger(profile_id(spec)+"_after_case")
    result = compare_cleanup_ledgers(baseline,final)
    require(result["passed"], f"Case did not clean up: {result}")
    case["cleanup_accounting"] = result

def profiles_stage():
    specs = sorted(profile_specs(),key=lambda s:(s["q"],s["role"],s["batch"],s["length"],s["phase"]))
    for index,spec in enumerate(specs):
        cid = profile_id(spec)
        if passed_case(cid) is not None:
            print("Reuse",cid,flush=True)
            continue
        case = new_case(cid,"profile",spec=spec,layer_order_sha256=canonical_hash(ORDER))
        finish_case(case,profile_case)
        print(f"Profile {index+1}/{len(specs)} saved: {cid}",flush=True)
    reset_fp16()
    return {"cases":len(specs),"fit_q":CFG["fit_q"],"validation_q":CFG["validation_q"],
        "measured_timing_trials":len(specs)*CFG["trials"],"memory_trials":len(specs)*CFG["memory_trials"],
        "supported_grid":{"batch":CFG["batches"],"context":CFG["lengths"]},"endpoints":"targeted anchor checks only"}

run_stage("profiles",profiles_stage)


## 11. Validate interpolation and verify final original FP16 state

Fit timing only from the fitting q values. Test q=10,18 at matching phase/batch/context and kernel dispatch;
no batch/context interpolation or extrapolation is attempted. Predeclared operation checks are WAPE ≤10%
and maximum per-case relative error ≤20%. These are separate from end-to-end simulator TTFT/latency tests.

If interpolation needs refinement, successful raw measurements remain valid and the result explicitly says so.
The final check restores every original FP16 layer and compares fresh FP16 logits.


In [ ]:
def completed_profile_cases():
    result = []
    for p in sorted((OUT/"cases").glob("*.json")):
        c = json.loads(p.read_text())
        if c["kind"]=="profile" and c["status"]=="PASS":
            result.append(c)
    return result



def interpolation_stage():
    summaries = profile_summary_rows(completed_profile_cases())
    rows = interpolation_rows(summaries,CFG["fit_q"])
    expected = len(CFG["validation_q"])*len(CFG["batches"])*len(CFG["lengths"])*2*2
    require(len(rows)==expected, "Interpolation validation grid is incomplete")
    groups = []
    for metric in ["gpu_operation_ms_median","synchronized_host_operation_ms_median"]:
        for phase in ["prefill","decode"]:
            subset = [r for r in rows if r["metric"]==metric and r["phase"]==phase]
            wape = 100*sum(r["absolute_error_ms"] for r in subset)/sum(r["measured_ms"] for r in subset)
            max_error = max(r["absolute_relative_error_percent"] for r in subset)
            groups.append({"metric":metric,"phase":phase,"cases":len(subset),"WAPE_percent":wape,
                "P95_case_relative_error_percent":quantile([r["absolute_relative_error_percent"] for r in subset],.95),
                "max_case_relative_error_percent":max_error,
                "within_predeclared_operation_limits":wape<=CFG["operation_interpolation_wape_limit_percent"]
                    and max_error<=CFG["operation_interpolation_max_case_error_percent"]})
    diagnostic = {"method":"linear in q, same phase/batch/context and dispatch only; no batch/context extrapolation",
        "status":"WITHIN_TESTED_OPERATION_LIMITS" if all(g["within_predeclared_operation_limits"] for g in groups) else "NEEDS_REFINEMENT",
        "fit_q":CFG["fit_q"],"heldout_q":CFG["validation_q"],"groups":groups,
        "WAPE_limit_percent":CFG["operation_interpolation_wape_limit_percent"],
        "max_case_error_limit_percent":CFG["operation_interpolation_max_case_error_percent"],
        "limits_scope":"Engineering checks on operation medians, not the methodology's end-to-end P95 TTFT/latency targets",
        "memory_policy":"Observed peaks only; these timing checks do not validate memory interpolation or safety reserves",
        "unvalidated":"Other unmeasured q values, unseen batch/context shapes, mixed batches and full simulator replay"}
    atomic_json(OUT/"interpolation_validation.json",diagnostic)
    save_csv(OUT/"interpolation_residuals.csv",rows)
    return diagnostic

run_stage("interpolation",interpolation_stage)

def final_restoration_stage():
    reset_fp16()
    for j in range(28):
        verify_layer("fp16",j)
    actual = selected_logits(DATA["calibration"][0])
    require(torch.equal(actual,ORIGINAL_TRACE), "Final fresh FP16 selected logits differ")
    assert_native()
    ledger = allocation_ledger("final_FP16_no_requests")
    require(ledger["valid"], "Invalid final allocation ledger")
    result = {"all_original_tensors_exact":True,"fresh_FP16_selected_logits_exact":True,
        "native_callable_restored":True,"awq_layers":sorted(ACTIVE_AWQ),"allocation_ledger":ledger["report_file"],
        "model_tensor_storage_bytes":ledger["model_tensor_storage_bytes"]}
    atomic_json(OUT/"final_restoration.json",result)
    return result

run_stage("final_restoration",final_restoration_stage)


## 12. Export the complete or partial result ZIP

Run this cell even if an earlier GPU cell failed. Failed attempts remain evidence and never enter summaries.
Download the displayed `lis_intermediate_<timestamp>.zip` or use Kaggle's output file panel. Send the full ZIP
for review. Keep the run folder, including scratch cache, if you intend to resume in Kaggle.

Primary outputs are `layer_order.json`, `configuration_footprints.csv`, `operation_summary.csv`, raw trials,
and the interpolation diagnostic. The README describes schemas, scope, source hashes and reproduction.


In [ ]:
def audit_cases(cases):
    errors = []
    expected = {f"static_layer_{j:02d}" for j in range(28)}
    order_path = OUT/"layer_order.json"
    record = json.loads(order_path.read_text()) if order_path.exists() else {}
    order = record.get("order",[])
    if len(order)!=28 or set(order)!=set(range(28)) or record.get("status")!="FROZEN":
        errors.append("Layer order is not fully frozen")
    else:
        if record.get("content_sha256")!=canonical_hash({k:v for k,v in record.items() if k!="content_sha256"}):
            errors.append("Frozen order hash mismatch")
        for q in range(28):
            expected |= {f"greedy_q{q:02d}_layer_{j:02d}" for j in range(28) if j not in order[:q]}
    def pid(role,q,phase,b,n):
        return f"profile_{role}_q{q:02d}_{phase}_b{b}_n{n}"
    for role,qs in [("fit",CFG["fit_q"]),("validation",CFG["validation_q"])]:
        expected |= {pid(role,q,phase,b,n) for q in qs for phase in ["prefill","decode"] for b in CFG["batches"] for n in CFG["lengths"]}
    expected |= {pid("anchor",q,phase,b,n) for q in [0,28] for phase in ["prefill","decode"] for b,n in CFG["anchor_shapes"]}
    completed = {}
    for case in cases:
        if case["status"]!="PASS":
            continue
        cid = case["case_id"]
        if cid in completed:
            errors.append(f"Duplicate completed attempt {cid}")
        completed[cid] = case
        if case["configuration_sha256"]!=CONFIG_HASH:
            errors.append(f"Case configuration differs: {cid}")
        if case["kind"] in ["static","greedy"]:
            if len(case["rows"])!=CFG["calibration_examples"]:
                errors.append(f"Wrong calibration example count: {cid}")
            fields = ["mds_tokens"] if case["kind"]=="greedy" else ["lts_tokens","lrs_tokens"]
            for row in case["rows"]:
                for name in fields:
                    values = row[name]
                    if len(values)!=CFG["score_positions_per_example"] or not all(math.isfinite(v) and -1<=v<=1 for v in values):
                        errors.append(f"Invalid cosine samples: {cid}")
            if case["kind"]=="static":
                for metric in ["lts","lrs"]:
                    expected_mean = statistics.fmean(x for row in case["rows"] for x in row[metric+"_tokens"])
                    if not math.isclose(case[metric],expected_mean,abs_tol=1e-12):
                        errors.append(f"Wrong static aggregate: {cid}")
            else:
                expected_mds = statistics.fmean(x for row in case["rows"] for x in row["mds_tokens"])
                expected_lis = .25*case["lts"]+.25*case["lrs"]+.5*expected_mds
                if not math.isclose(case["mds"],expected_mds,abs_tol=1e-12) or not math.isclose(case["lis"],expected_lis,abs_tol=1e-12):
                    errors.append(f"Wrong greedy aggregate: {cid}")
        elif case["kind"]=="profile":
            counts = {"warmup":0,"timing":0,"memory":0}
            for row in case["rows"]:
                key = "warmup" if row["warmup"] else row["instrumentation"]
                counts[key] += 1
                if not row["all_tracked_request_tensors_released"]:
                    errors.append(f"Request state retained: {cid}")
                if len(order)==28 and row["awq_layers"]!=sorted(order[:row["q"]]):
                    errors.append(f"Wrong active layer set: {cid}")
            if counts!={"warmup":CFG["warmups"],"timing":CFG["trials"],"memory":CFG["memory_trials"]}:
                errors.append(f"Wrong profiling counts: {cid}")
            if not case["cleanup_accounting"]["passed"]:
                errors.append(f"Cleanup not verified: {cid}")
    if len(order)==28:
        for q,chosen in enumerate(order):
            candidates = [completed.get(f"greedy_q{q:02d}_layer_{j:02d}") for j in range(28) if j not in order[:q]]
            if any(c is None for c in candidates):
                continue
            if min(candidates,key=lambda c:(-c["lis"],c["layer"]))["layer"]!=chosen:
                errors.append(f"Wrong greedy winner at q={q}")
            if any(c["prefix"]!=order[:q] for c in candidates):
                errors.append(f"Wrong candidate prefix at q={q}")
    footprints = json.loads((OUT/"configuration_footprints.json").read_text()) if (OUT/"configuration_footprints.json").exists() else []
    if sorted(r["q"] for r in footprints)!=list(range(29)):
        errors.append("Missing configuration footprint records")
    missing = sorted(expected-set(completed))
    unexpected = sorted(set(completed)-expected) if len(order)==28 else []
    return {"expected_cases_when_order_complete":len(expected) if len(order)==28 else None,
        "completed_cases":len(completed),"missing_cases":missing,"unexpected_cases":unexpected,
        "frozen_order_complete":len(order)==28,"errors":errors,
        "pass":not(errors or missing or unexpected)}

def export_results():
    cases = [json.loads(p.read_text()) for p in sorted((OUT/"cases").glob("*.json"))]
    static,greedy,static_raw,greedy_raw,operation_raw,memory = [],[],[],[],[],[]
    profiles = []
    for case in cases:
        accepted = case["status"]=="PASS"
        shared = {"case_id":case["case_id"],"attempt":case["attempt"],"accepted_case":accepted}
        if case["kind"]=="static":
            for row in case["rows"]:
                static_raw.append({**shared,"layer":case["layer"],**row})
            if accepted:
                static.append({**shared,"layer":case["layer"],"lts":case["lts"],"lrs":case["lrs"],
                    "scored_token_count":case["scored_token_count"],"fp16_replay_max_absolute_difference":case["fp16_replay_max_absolute_difference"]})
        elif case["kind"]=="greedy":
            for row in case["rows"]:
                greedy_raw.append({**shared,"q":case["q"],"layer":case["layer"],"prefix":case["prefix"],**row})
            if accepted:
                greedy.append({**shared,**{k:case[k] for k in ["q","layer","prefix","lts","lrs","mds","lis"]}})
        elif case["kind"]=="profile":
            if accepted:
                profiles.append(case)
            for row in case["rows"]:
                operation_raw.append({**shared,**{k:v for k,v in row.items() if k!="memory_points"}})
                for point in row["memory_points"]:
                    memory.append({**shared,**{k:row[k] for k in ["q","phase","batch","length","role","warmup","repeat","instrumentation"]},**point})
    save_csv(OUT/"static_scores.csv",static)
    save_csv(OUT/"static_token_scores.csv",static_raw)
    save_csv(OUT/"greedy_scores.csv",greedy)
    save_csv(OUT/"greedy_token_scores.csv",greedy_raw)
    save_csv(OUT/"operation_trials.csv",operation_raw)
    save_csv(OUT/"operation_memory_snapshots.csv",memory)
    save_csv(OUT/"operation_summary.csv",profile_summary_rows(profiles))
    if (OUT/"configuration_footprints.json").exists():
        save_csv(OUT/"configuration_footprints.csv",json.loads((OUT/"configuration_footprints.json").read_text()))
    audit = audit_cases(cases)
    atomic_json(OUT/"coverage_audit.json",audit)
    complete = all(c["status"]=="PASS" for c in REPORT["checks"].values()) and audit["pass"]
    interpolation_path = OUT/"interpolation_validation.json"
    interpolation = json.loads(interpolation_path.read_text()) if interpolation_path.exists() else {}
    REPORT["completion_status"] = ("COMPLETE_INTERPOLATION_WITHIN_TESTED_LIMITS" if interpolation.get("status")=="WITHIN_TESTED_OPERATION_LIMITS"
                                    else "COMPLETE_INTERPOLATION_NEEDS_REFINEMENT") if complete else "INCOMPLETE_OR_FAILED"
    REPORT["coverage"] = audit
    REPORT["export_utc"] = datetime.now(timezone.utc).isoformat()
    source = base64.b64decode(PROFILER_SOURCE_BASE64)
    require(hashlib.sha256(source).hexdigest()==CODE_SHA256,"Distributed source integrity failure")
    (OUT/"distributed_notebook_source.py").write_bytes(source)
    (OUT/"README.md").write_text(RESULT_README)
    (OUT/"reproduce.py").write_text(REPRODUCE_SCRIPT)
    function_sources = []
    for name in ["cosine_tokens","greedy_stage","greedy_candidate_case","static_layer_case","capture_fp16_activations",
                 "replay_selected","operation_trial","checked_trial","profile_case","interpolation_stage",
                 "switch_once","allocation_ledger","compare_cleanup_ledgers"]:
        fn = globals().get(name)
        if fn is None:
            function_sources.append(f"# {name} not defined before this partial export")
            continue
        try:
            function_sources.append(inspect.getsource(fn))
        except (OSError,TypeError):
            function_sources.append(f"# Source unavailable for {name}; use the saved notebook version")
    current = "\n\n".join(function_sources).encode()
    (OUT/"executed_function_sources.py").write_bytes(current)
    REPORT["executed_function_sources_sha256"] = hashlib.sha256(current).hexdigest()
    atomic_json(OUT/"errors.json",{
        "current_stages":{k:v for k,v in REPORT["checks"].items() if v["status"] not in ["PASS","NOT_RUN"]},
        "case_attempts":[{"case_id":c["case_id"],"status":c["status"],"error":c.get("error")} for c in cases if c["status"]!="PASS"]})
    save_report()
    hashes = {str(p.relative_to(OUT)):{"bytes":p.stat().st_size,"sha256":hashlib.sha256(p.read_bytes()).hexdigest()}
              for p in sorted(OUT.rglob("*")) if p.is_file() and not p.is_relative_to(CACHE)
              and p.name!="file_hashes.json" and not p.name.endswith(".tmp")}
    atomic_json(OUT/"file_hashes.json",hashes)
    archive = archive_outputs()
    print("Status:",REPORT["completion_status"])
    print("Completed cases:",audit["completed_cases"])
    print("ZIP:",archive)
    print("ZIP SHA256:",hashlib.sha256(archive.read_bytes()).hexdigest())
    print("Activation scratch is excluded from this ZIP; keep the writable run folder if resuming in Kaggle.")
    try:
        from IPython.display import FileLink,display
        display(FileLink(str(archive.relative_to(WORKING))))
    except Exception:
        pass
    return archive

RESULT_ZIP = export_results()


The new algorithm/replay/validation logic passed 11 CPU fixtures, including all 406 candidate decisions in
a state-dependent toy model, exact tie-breaking, full-context LRS replay, disjoint data windows, frozen-path
resume, and partial export. The shared swapping executor was previously exercised on your T4 run.
**This new notebook still requires its first GPU execution on Kaggle.** Similarity scores do not establish
semantic output quality, and these operation checks do not validate an elastic allocator or scheduler gains.
